# TFM Sistema Multiagente de Predicción de Churn

Este notebook contiene el desarrollo del sistema agéntico (CrewAI) que se compara contra el modelo de Machine Learning entrenado en el notebook `ML_MODEL.ipynb`.

**Estructura del notebook:**
1. Setup del entorno (Drive, instalación de librerías)
2. Carga del modelo de ML y artefactos (scaler, feature order)
3. Carga librería crewai para agentes
5. Agente Predictor (tool + agente + tarea)
6. Agente Analista
7. Agente Recomendador



## 1. Setup del entorno

In [1]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta de la carpeta de trabajo en Drive
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/TFM - MULTIAGENTES'
os.chdir(WORK_DIR)

print(f"Directorio de trabajo activado: {os.getcwd()}")

Mounted at /content/drive
Directorio de trabajo activado: /content/drive/MyDrive/Colab Notebooks/TFM - MULTIAGENTES


In [2]:
# !pip install -q "crewai[litellm]" litellm --upgrade
# !pip install -q -U crewai
!pip install crewai==1.15.21 litellm==1.101.0 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

In [ ]:
# from importlib.metadata import version
# print("crewai:", version("crewai"))
# print("litellm:", version("litellm"))

### Configuración de la API Key

**Importante:** no dejes la API key escrita directamente en una celda del notebook. Si compartes o subes este notebook (a un repo, a Drive compartido, etc.), la clave queda expuesta. En Colab, usa **Secrets** (icono de la llave en el panel izquierdo) para guardarla de forma segura, y recupérala así:

In [3]:
from google.colab import userdata
import os

# La API key se guarda en Colab Secrets (icono de llave, panel izquierdo),
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API2')
#os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API3')

## 2. Carga del modelo de ML y artefactos

Cargamos el modelo entrenado junto con el `scaler` y el orden de columnas (`feature_order`) generados en el notebook de entrenamiento. Estos tres artefactos son los que permiten reproducir exactamente el mismo pipeline de preprocesado fuera del notebook original.

In [2]:
import joblib
import pandas as pd

model = joblib.load('best_model_lgbm.pkl')
scaler = joblib.load('scaler.pkl')
FEATURE_ORDER = joblib.load('feature_order.pkl')

## 3. Importación para agentes

In [5]:
import os
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool

# Workaround para el bug de cache_breakpoint con Groq
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

## 5. Agente Predictor

Este agente tiene una única responsabilidad: llamar al modelo de ML a través de la tool `predecir_churn_tool` y devolver el nivel de riesgo de churn de un cliente, **sin interpretar ni mencionar ninguna de sus variables** (edad, país, productos, etc.). Esa interpretación es responsabilidad del agente Analista, que sí trabajará con el contexto completo del cliente y del histórico de `train`.

### Función `predict_churn`

Recibe los datos crudos de un cliente (tal como vienen en el CSV original de Kaggle, sin variables derivadas ni encoding) y aplica internamente todo el pipeline necesario para llamar al modelo: creación de features derivadas (`HasZeroBalance`, `ProductRisk`), encoding de categóricas, escalado y reordenación de columnas.

In [6]:
# Columnas que debe tener cada fila (cliente) de entrada
REQUIRED_COLS = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure',
                  'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                  'EstimatedSalary']

# Columnas numéricas que necesitan escalado
NUM_COLS = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']

# Valores válidos de las columnas categóricas
VALID_GEOGRAPHY = ['France', 'Germany', 'Spain']
VALID_GENDER = ['Male', 'Female']


def predict_churn(customer_data: dict) -> dict:
    """
    Recibe los datos crudos de un cliente (como en el CSV original) y devuelve
    la predicción de churn del modelo de ML.

    customer_data debe incluir las claves:
    CreditScore, Geography, Gender, Age, Tenure, Balance,
    NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary
    """
    missing = [c for c in REQUIRED_COLS if c not in customer_data]
    if missing:
        raise ValueError(f"Faltan columnas requeridas para predecir: {missing}")

    if customer_data['Geography'] not in VALID_GEOGRAPHY:
        raise ValueError(f"Geography inválida: {customer_data['Geography']}")

    if customer_data['Gender'] not in VALID_GENDER:
        raise ValueError(f"Gender inválido: {customer_data['Gender']}")

    df = pd.DataFrame([customer_data])

    # 1. Eliminar columnas irrelevantes si vienen incluidas (identificadores, target)
    df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'], errors='ignore')

    # 2. Features derivadas
    df['HasZeroBalance'] = (df['Balance'] == 0).astype(int)
    df['ProductRisk'] = df['NumOfProducts'].map({1: 0, 2: 0, 3: 1, 4: 1})

    # 3. Encoding
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df['Geography_Germany'] = (df['Geography'] == 'Germany').astype(int)
    df['Geography_Spain'] = (df['Geography'] == 'Spain').astype(int)
    df = df.drop(columns=['Geography'])

    # 4. Escalado (usando el scaler YA ajustado en train, solo transform)
    df[NUM_COLS] = scaler.transform(df[NUM_COLS])

    # 5. Reordenar columnas exactamente como en entrenamiento
    df = df[FEATURE_ORDER]

    # 6. Predicción
    pred = model.predict(df)[0]
    proba = model.predict_proba(df)[0][1]

    return {
        'churn_prediction': int(pred),
        'churn_probability': float(proba)
    }

### Validación de `predict_churn`

Confirmamos que la función reproduce exactamente las mismas predicciones que el modelo original calculó sobre `X_test` en el notebook de entrenamiento. Esto garantiza que el pipeline de preprocesado (encoding, features derivadas, escalado, orden de columnas) está correctamente replicado en este notebook, independiente del de entrenamiento.

In [ ]:
# Cargamos el test set crudo y las predicciones originales del notebook de entrenamiento
X_test_raw = pd.read_csv('X_test_raw.csv')
predicciones_originales = pd.read_csv('predicciones_best_lgbm.csv')

# Nota: X_test_raw se guardó con index=False, así que el emparejamiento entre
# ambos CSV es por posición de fila (válido porque ambos salen del mismo split
# y no se ha reordenado nada entre medias)

n_pruebas = 30
coincidencias = 0

for i in range(n_pruebas):
    fila = X_test_raw.iloc[i].to_dict()
    resultado = predict_churn(fila)

    pred_original = predicciones_originales.iloc[i]['pred_clase']
    proba_original = predicciones_originales.iloc[i]['pred_proba']

    coincide_pred = resultado['churn_prediction'] == pred_original
    coincide_proba = abs(resultado['churn_probability'] - proba_original) < 1e-6

    if coincide_pred and coincide_proba:
        coincidencias += 1
    else:
        print(f"⚠️ Fila {i}: función={resultado}, original=({pred_original}, {proba_original})")

print(f"\nCoincidencias: {coincidencias}/{n_pruebas}")


Coincidencias: 30/30


### 5.1. Tool: envoltorio de `predict_churn`

In [7]:
@tool("Predecir Churn de Cliente")
def predecir_churn_tool(CreditScore: int, Geography: str, Gender: str, Age: int,
    Tenure: int, Balance: float, NumOfProducts: int, HasCrCard: int,
    IsActiveMember: int, EstimatedSalary: float ) -> str:
    """
    ÚTIL ÚNICAMENTE PARA: calcular la probabilidad de abandono (churn) de un cliente
    bancario, usando un modelo de Machine Learning ya entrenado (LightGBM).

    Debes llamar SIEMPRE a esta tool cuando necesites predecir el riesgo de churn
    de un cliente. No inventes ni estimes el resultado por tu cuenta bajo ningún concepto.

    Recibe los datos crudos del cliente, usando EXACTAMENTE las
    mismas claves que se te han dado en la descripción de la tarea (CreditScore,
    Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard,
    IsActiveMember, EstimatedSalary), sin traducirlas ni cambiarles el nombre.
    Devuelve la predicción (0=no churn, 1=churn) junto con la probabilidad estimada.
    """
    customer_data = {
        'CreditScore': CreditScore,
        'Geography': Geography,
        'Gender': Gender,
        'Age': Age,
        'Tenure': Tenure,
        'Balance': Balance,
        'NumOfProducts': NumOfProducts,
        'HasCrCard': HasCrCard,
        'IsActiveMember': IsActiveMember,
        'EstimatedSalary': EstimatedSalary
    }

    resultado = predict_churn(customer_data)
    probabilidad_pct = resultado['churn_probability'] * 100 # Se calcula el % aquí fuera para evitar errores del LLM

    return (
        f"Predicción: {resultado['churn_prediction']} "
        f"(0=no abandona, 1=abandona) | "
        f"Probabilidad de churn: {probabilidad_pct:.2f}%"
    )

### 5.2. Definición del agente

In [8]:
llm_predictor = LLM(
    model="groq/openai/gpt-oss-20b",
    temperature=0.1,  # baja temperatura: queremos un resultado consistente, no creativo
)

agente_predictor = Agent(
    role='Agente Predictor de Churn',
    goal='Calcular el riesgo de abandono de un cliente usando exclusivamente el modelo de ML, sin interpretar ni analizar sus datos.',
    backstory=(
        'Eres un componente especializado y estrictamente limitado a una única función: '
        'llamar al modelo predictivo de churn para obtener un resultado numérico. '
        'SIEMPRE debes usar la tool disponible para obtener la predicción, nunca debes '
        'estimar o inventar un resultado por tu cuenta. '
        'NUNCA debes explicar por qué un cliente tiene ese riesgo, ni mencionar ninguna '
        'variable o dato del cliente (edad, país, productos, salario, etc.) en tu respuesta. '
        'Ese análisis lo realiza otro agente especializado, no tú. '
        'Tu única salida debe ser: nombre del cliente, predicción de churn y probabilidad exacta.'
    ),
    tools=[predecir_churn_tool],
    verbose=True,
    llm=llm_predictor
)

### 5.3. tarea

Función que genera la `Task` para un cliente concreto, para no repetir la descripción cada vez que se quiera evaluar a un cliente distinto.

In [ ]:
from pydantic import BaseModel #para forzar salida validada

class PrediccionChurn(BaseModel):
    cliente: str
    churn_prediction: int
    churn_probability_pct: float # 0-100%, tal cual lo devuelve la tool


def crear_tarea_prediccion(nombre_cliente: str, customer_data: dict) -> Task:
    return Task(
        description=(
            f'Cliente: {nombre_cliente}.\n'
            f'Datos del cliente: {customer_data}\n\n'
            'Llama a la tool de predicción de churn con estos datos.'
            'No añadas ninguna explicación, justificación ni mención a las variables del cliente.'
        ),
        expected_output=('Un objeto con: cliente (el nombre), churn_prediction (0 o 1, tal cual lo '
            'devuelve la tool) y churn_probability_pct % (el porcentaje de riesgo de churn, '
            'tal cual lo devuelve la tool, sin recalcularlo ni redondearlo tú).'),
        agent=agente_predictor,
        output_pydantic=PrediccionChurn,
    )

### 5.4. Prueba end-to-end del Agente Predictor

Ejecutamos el agente completo (Crew con un único agente y una única tarea) sobre una fila real del conjunto de test.

In [ ]:
X_test_raw = pd.read_csv('X_test_raw.csv')

fila = X_test_raw.iloc[8]
nombre_cliente = fila['Surname']
customer_data = fila.drop(labels=['RowNumber', 'CustomerId', 'Surname']).to_dict()

tarea = crear_tarea_prediccion(nombre_cliente, customer_data)

mi_sistema = Crew(
    agents=[agente_predictor],
    tasks=[tarea],
    process=Process.sequential
)

resultado = await mi_sistema.kickoff_async()
print(resultado)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Predictor de Churn                                                                               │
│                                                                                                                 │
│  Task: Cliente: Lo.                                                                                             │
│  Datos del cliente: {'CreditScore': 505, 'Geography': 'France', 'Gender': 'Male', 'Age': 49, 'Tenure': 7,       │
│  'Balance': 80001.23, 'NumOfProducts': 1, 'HasCrCard': 0, 'IsActiveMember': 0, 'EstimatedSalary': 135180.11}    │
│                                                                                                                 │
│  Llama a la tool de predicción de churn con estos datos.No añadas ninguna explicación, justificación ni         │
│  mención a las variables del cliente.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool predecir_churn_de_cliente executed with result: Predicción: 1 (0=no abandona, 1=abandona) | Probabilidad de churn: 77.79%...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Predictor de Churn                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"cliente":"Lo","churn_prediction":1,"churn_probability_pct":77.79}                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

cliente='Lo' churn_prediction=1 churn_probability_pct=77.79




## 6. Agente Analista

### 6.1 Cargar dataset train y test. Calcular estadísticos de referencia.

In [3]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CARGA DE DATOS DE ENTRENAMIENTO Y TEST
# ==========================================

# Cargamos el conjunto de entrenamiento (features y target)
X_train_raw = pd.read_csv('X_train_raw.csv')
y_train_raw = pd.read_csv('y_train_raw.csv')
assert len(X_train_raw) == len(y_train_raw)

# Creamos df_train combinando ambos para que las Tools puedan calcular estadísticas
df_train = X_train_raw.copy()
if 'Exited' not in df_train.columns:
    df_train['Exited'] = y_train_raw['Exited'] if 'Exited' in y_train_raw.columns else y_train_raw.iloc[:, 0]

# Cargamos el conjunto de TEST (solo features, ciego)
X_test_raw = pd.read_csv('X_test_raw.csv')

print(f"✅ Dataframe de entrenamiento listo para Tools: {df_train.shape}")
print(f"✅ Dataframe de test listo para inferencia: {X_test_raw.shape}")

✅ Dataframe de entrenamiento listo para Tools: (8000, 14)
✅ Dataframe de test listo para inferencia: (2000, 13)


In [10]:
# --------------------------------------------------
# Estadísticos de referencia (calculados sobre df_train)
# ---------------------------------------------------

def calcular_referencias(df: pd.DataFrame) -> dict:
    """Calcula estadísticos descriptivos de df_train para dar contexto objetivo
    al analista, evitando que el LLM tenga que 'adivinar' si un valor
    es alto/bajo sin referencia. No decide nada, solo describe la distribución
    de la base para que el agente pueda juzgar qué datos del cliente merecen
    una consulta."""
    referencias = {}
    for col in ['Balance', 'Age']:
        referencias[col] = {
            'media': df[col].mean(),
            'std': df[col].std(),
            'q25': df[col].quantile(0.25),
            'q75': df[col].quantile(0.75),
        }
    return referencias

referencias_train = calcular_referencias(df_train)

referencias_texto = "\n".join(
    f"- {col}: media={r['media']:.1f}, rango típico (Q25-Q75)=[{r['q25']:.1f} - {r['q75']:.1f}]"
    for col, r in referencias_train.items() if col in ('Balance', 'Age')
)
print(referencias_texto)

# -------------------------------------------------------------------------
# --- Referencias que usan las tools para contextualizar sus resultados
# ------------------------------------------------------------------------
BASE_CHURN = df_train['Exited'].mean() * 100                      # tasa media de toda la base
Q_BALANCE_POS = df_train.loc[df_train['Balance'] > 0, 'Balance'].quantile([.25, .50, .75]).values
print(f"- Tasa de abandono media de la base: {BASE_CHURN:.1f}%", "\n")


# -----------------------------------------------------------------------
# Anclas para los docstrings, derivadas de df_train.
# Describen DÓNDE se separa cada variable, nunca cuánto: la cifra exacta
# sigue siendo responsabilidad de la tool, así que el agente tiene que llamarla
# si quiere citarla. Al calcularse del propio train, se regeneran solas si los
# datos cambian, en vez de quedarse desfasadas escritas a mano.
# Se añaden para ayudar al agente a elegir estas herramientas.
# ----------------------------------------------------------------------

def _enumerar(items: list) -> str:
    if not items:
        return "ninguno"
    if len(items) == 1:
        return str(items[0])
    return ", ".join(str(i) for i in items[:-1]) + f" y {items[-1]}"


def _niveles_por_lift(tasas: dict, tamanos: dict, base: float, minimo_n: int = 100):
    """Separa los niveles de una variable en los que quedan por encima y por debajo de la
    media global, ignorando los de muestra insuficiente."""
    altos, bajos = [], []
    for nivel, tasa in tasas.items():
        if tamanos[nivel] < minimo_n:
            continue
        lift = tasa / base
        if lift >= 1.20:
            altos.append(nivel)
        elif lift <= 0.85:
            bajos.append(nivel)
    return altos, bajos


# --- País ---
_paises_altos, _paises_bajos = _niveles_por_lift(
    (df_train.groupby('Geography')['Exited'].mean() * 100).to_dict(),
    df_train.groupby('Geography').size().to_dict(), BASE_CHURN)

# --- Edad, en tramos de diez años ---
_banda_edad = (df_train['Age'] // 10 * 10)
_edad_altas, _edad_bajas = _niveles_por_lift(
    (df_train.groupby(_banda_edad)['Exited'].mean() * 100).to_dict(),
    df_train.groupby(_banda_edad).size().to_dict(), BASE_CHURN)
_edad_altas = [f"{int(b)}-{int(b)+9}" for b in sorted(_edad_altas)]
_edad_bajas = [f"{int(b)}-{int(b)+9}" for b in sorted(_edad_bajas)]

# --- Productos ---
_tasas_prod = df_train.groupby('NumOfProducts')['Exited'].mean() * 100
_prod_estable = int(_tasas_prod.idxmin())
_prod_altos, _ = _niveles_por_lift(
    _tasas_prod.to_dict(), df_train.groupby('NumOfProducts').size().to_dict(),
    BASE_CHURN, minimo_n=30)


DOC_CONTEXTO = (
    "Devuelve la tasa histórica de abandono del país del cliente y de su franja de edad "
    "(tramos de diez años), ambas comparadas con la media global. "
    f"Los mercados no se comportan igual: {_enumerar(_paises_altos)} se sitúa por encima de la "
    f"media y {_enumerar(_paises_bajos)} por debajo. Por edad, las franjas de "
    f"{_enumerar(_edad_altas)} años quedan por encima y las de {_enumerar(_edad_bajas)} años por "
    "debajo. Consúltala cuando el cliente pertenezca a alguno de los grupos que se sitúan por "
    "encima. El banco no puede intervenir sobre estas variables: "
    "el resultado dimensiona la gravedad del caso, no señala una palanca."
)

DOC_PRODUCTOS = (
    "Devuelve la tasa histórica de abandono de los clientes con el mismo número de productos, "
    "comparada con la media global. La relación no es monótona y contradice la intuición de que "
    f"más productos implican más fidelidad: contratar {_prod_estable} productos es el punto más "
    f"estable de la escala, mientras que {_enumerar(_prod_altos)} producto(s) se sitúan por "
    "encima de la media."
)

print(DOC_CONTEXTO, "\n")
print(DOC_PRODUCTOS)

- Balance: media=76381.2, rango típico (Q25-Q75)=[0.0 - 127633.2]
- Age: media=38.9, rango típico (Q25-Q75)=[32.0 - 44.0]
- Tasa de abandono media de la base: 20.4% 

Devuelve la tasa histórica de abandono del país del cliente y de su franja de edad (tramos de diez años), ambas comparadas con la media global. Los mercados no se comportan igual: Germany se sitúa por encima de la media y France y Spain por debajo. Por edad, las franjas de 40-49, 50-59 y 60-69 años quedan por encima y las de 20-29, 30-39 y 70-79 años por debajo. Consúltala cuando el cliente pertenezca a alguno de los grupos que se sitúan por encima. El banco no puede intervenir sobre estas variables: el resultado dimensiona la gravedad del caso, no señala una palanca. 

Devuelve la tasa histórica de abandono de los clientes con el mismo número de productos, comparada con la media global. La relación no es monótona y contradice la intuición de que más productos implican más fidelidad: contratar 2 productos es el punto más 

### 6.2 Tools

In [11]:
from functools import wraps

# =========================================================
# Registro de llamadas a tools (instrumentación, no entra en el prompt).
# Permite contrastar qué herramientas usó realmente el agente frente a las
# que dice haber usado en su respuesta final.
# =========================================================

registro_llamadas = []

def registrar(nombre: str):
    """Envuelve una tool para dejar constancia de su llamada y de su resultado.
    @wraps conserva firma y docstring, de modo que CrewAI sigue generando el mismo
    esquema de parámetros que si la función no estuviera decorada."""
    def decorador(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            resultado = func(*args, **kwargs)
            registro_llamadas.append({'tool': nombre, 'resultado': resultado})
            return resultado
        return wrapper
    return decorador


def iniciar_registro():
    """Vacía el registro. Llámala justo antes de cada kickoff."""
    registro_llamadas.clear()


def tools_llamadas() -> list:
    """Nombres de las tools realmente invocadas, en orden y sin repetidos."""
    vistas = []
    for llamada in registro_llamadas:
        if llamada['tool'] not in vistas:
            vistas.append(llamada['tool'])
    return vistas

In [12]:
from crewai.tools import tool

#--- Formato común de salida de las tools ---
# Etiquetas en la salida para el LLM
def _formato_segmento(descripcion: str, n: int, tasa: float) -> str:
    """Devuelve siempre el segmento comparado con la base global.
    Un porcentaje solo no es interpretable sin saber cuál es la media;
    el cociente frente a la base sí lo es."""
    if n == 0:
        return f"{descripcion}: no hay clientes con este perfil en la base histórica."
    lift = tasa / BASE_CHURN
    if lift >= 1.50:
        lectura = "muy por encima de la media"
    elif lift >= 1.20:
        lectura = "por encima de la media"
    elif lift <= 0.60:
        lectura = "muy por debajo de la media"
    elif lift <= 0.85:
        lectura = "por debajo de la media"
    else:
        lectura = "en línea con la media"
    aviso = " [muestra pequeña: interpretar con cautela]" if n < 100 else ""
    return (f"{descripcion} (n={n}): churn {tasa:.1f}% vs {BASE_CHURN:.1f}% base "
            f"→ x{lift:.2f}, {lectura}{aviso}")

# --- Tools del Analista Palanca ---
def _consultar_riesgo_productos(num_productos: int) -> str:
    try:
        seg = df_train[df_train['NumOfProducts'] == num_productos]
        return _formato_segmento(f"Clientes con {num_productos} producto(s)",
                                 len(seg), seg['Exited'].mean() * 100)
    except Exception as e:
        return f"Error al consultar el número de productos: {e}"

@tool("Actividad")
@registrar("Score de actividad y abandono por segmento")
def consultar_actividad_vinculacion(is_active_member: int, has_crcard: int) -> str:
    """Devuelve la tasa histórica de abandono del segmento formado por el estado de actividad
    (IsActiveMember) y si tiene tarjeta (HasCrCard), comparada con la media global. La
    actividad separa la base en dos grupos con tasas muy distintas; en clientes activos el
    resultado queda cerca de la media global."""
    try:
        seg = df_train[(df_train['IsActiveMember'] == is_active_member) &
                       (df_train['HasCrCard'] == has_crcard)]
        estado = "activos" if is_active_member == 1 else "inactivos"
        tarjeta = "con tarjeta" if has_crcard == 1 else "sin tarjeta"
        return _formato_segmento(f"Clientes {estado} y {tarjeta}",
                                 len(seg), seg['Exited'].mean() * 100)
    except Exception as e:
        return f"Error al evaluar la vinculación: {e}"

# --- Tools del Analista Contexto ---
@tool("Balance")
@registrar("Percentil de balance")
def calcular_percentil_balance(balance: float) -> str:
    """Sitúa el saldo del cliente en la distribución real de la base y devuelve la tasa histórica
    de abandono de su tramo de saldo, comparada con la media global. Una cifra en euros no es
    interpretable sin saber qué posición ocupa; los extremos (saldo nulo o muy alto) son los que
    más se separan de la media."""
    try:
        if balance == 0:
          seg = df_train[df_train['Balance'] == 0]
          return _formato_segmento("Clientes con saldo 0 €", len(seg), seg['Exited'].mean() * 100)

        percentil = (df_train['Balance'] < balance).mean() * 100
        q1, q2, q3 = Q_BALANCE_POS
        if balance <= q1:
            seg = df_train[(df_train['Balance'] > 0) & (df_train['Balance'] <= q1)]
            tramo = "primer cuartil de saldo positivo"
        elif balance <= q2:
            seg = df_train[(df_train['Balance'] > q1) & (df_train['Balance'] <= q2)]
            tramo = "segundo cuartil de saldo positivo"
        elif balance <= q3:
            seg = df_train[(df_train['Balance'] > q2) & (df_train['Balance'] <= q3)]
            tramo = "tercer cuartil de saldo positivo"
        else:
            seg = df_train[df_train['Balance'] > q3]
            tramo = "cuarto cuartil de saldo positivo"

        return (f"Saldo de {balance:,.2f} € → percentil {percentil:.1f} de la base. "
                  + _formato_segmento(f"Clientes del {tramo}", len(seg), seg['Exited'].mean() * 100))
    except Exception as e:
        return f"Error al calcular el percentil de balance: {e}"

def _consultar_contexto_demografico(geography: str, age: int) -> str:
    try:
        seg_geo = df_train[df_train['Geography'] == geography]
        inf = (int(age) // 10) * 10
        sup = inf + 9
        seg_edad = df_train[(df_train['Age'] >= inf) & (df_train['Age'] <= sup)]

        linea_pais = _formato_segmento(f"Clientes de {geography}", len(seg_geo),
                                       seg_geo['Exited'].mean() * 100)
        linea_edad = _formato_segmento(f"Clientes de {inf}-{sup} años", len(seg_edad),
                                       seg_edad['Exited'].mean() * 100)
        return f"[País] {linea_pais}\n[Edad] {linea_edad}"
    except Exception as e:
        return f"Error al consultar el contexto demográfico: {e}"


# Los docstrings se asignan antes de decorar porque se construyen con f-strings a partir
# de df_train, y un f-string no puede ser docstring literal.
_consultar_riesgo_productos.__doc__ = DOC_PRODUCTOS
_consultar_contexto_demografico.__doc__ = DOC_CONTEXTO

consultar_riesgo_productos = tool("Productos")(
    registrar("Riesgo por número de productos")(_consultar_riesgo_productos))

consultar_contexto_demografico = tool("Contexto")(
    registrar("Contexto demográfico")(_consultar_contexto_demografico))

### 6.3 Definición del agente

In [13]:
llm_analista = LLM(
    model="groq/openai/gpt-oss-20b",
    temperature=0.0,  # bajamos temperatura: decisiones consistentes, no creatividad
    seed = 42,
    reasoning_effort="medium"
)

agente_analista = Agent(
    role='Analista de Negocio Senior',
    goal='Caracterizar el perfil de un cliente bancario frente al histórico, citando solo la '
         'evidencia que realmente cambia la lectura del caso.',
        backstory=(
        'Eres un analista de negocio senior especializado en cartera de clientes bancarios.\n\n'
        'No calculas ni recuerdas cifras: cualquier tasa o percentil de tu análisis debe proceder '
        'de una herramienta. Si no la has consultado, no tienes el dato.\n\n'
        'Tu informe se juzga por la calidad de la evidencia que cita, no por su extensión. Antes '
        'de usar una herramienta, valora si su resultado puede añadir algo a tu conclusión sobre '
        'ESTE cliente; si solo va a confirmar algo ya evidente en sus datos, no la uses. Dos '
        'señales bien elegidas valen más que cinco de las que tres son irrelevantes, y dejar una '
        'herramienta sin usar es una decisión correcta.\n\n'
        'Referencias de la distribución de la base, para juzgar si un dato del cliente es atípico '
        f'(describen la base, no el riesgo de abandono):\n{referencias_texto}\n\n'
        'No propones acciones de retención ni predices si el cliente abandonará: solo caracterizas '
        'su perfil frente al histórico.'
    ),
    max_iter=8,
    tools=[
        calcular_percentil_balance,
        consultar_riesgo_productos,
        consultar_actividad_vinculacion,
        consultar_contexto_demografico,
    ],
    verbose=True,
    llm=llm_analista
)

### 6.4 Tarea

In [ ]:
# Establecemos la forma en la que el llm nos devuelve la salida de la tarea
from typing import List, Literal
from pydantic import BaseModel, Field


class AnalisisCliente(BaseModel):
    cliente: str
    senales: List[str] = Field(..., description="Una entrada por herramienta usada, con el nombre "
                                               "de la herramienta y el resultado exacto que devolvió")
    orientacion: Literal[
        "riesgo_por_encima_de_la_media",
        "riesgo_en_linea_con_la_media",
        "riesgo_por_debajo_de_la_media",
    ] = Field(..., description="Hacia dónde inclina el conjunto de señales. Debes elegir una: "
                              "no existe la opción 'mixto'")
    conclusion: str = Field(..., description="1-2 frases justificando la orientación, sin proponer "
                                            "acciones de retención ni mencionar predicciones")

# Tarea
def analizar_cliente(nombre_cliente:str, cliente: dict):
    datos_formateados = "\n".join(f"- {k}: {v}" for k, v in cliente.items())

    return Task(
        description=(
            f"Analiza el perfil del cliente '{nombre_cliente}' frente al histórico de la base.\n"
            f"Datos del cliente:\n{datos_formateados}\n\n"
            f"Empieza por las variables de comportamiento del cliente con la entidad (productos "
            f"contratados y estado de actividad), que son las que más separan la base histórica. "
            f"Añade después las variables de perfil (saldo y contexto demográfico) solo si su "
            f"resultado puede cambiar la lectura del caso. Responde en español."
            f"Al elegir la orientación, pondera la intensidad de cada señal, no su número: "
            f"una señal muy alejada de la media (por ejemplo x4) pesa más que varias señales "
            f"ligeramente por debajo. Una sola señal extrema puede determinar la orientación."
        ),
            expected_output=(
            "Nombre del cliente, lista de señales consultadas con sus cifras exactas, "
                         "orientación global del caso y conclusión de 1-2 frases."
        ),
            output_pydantic=AnalisisCliente,
            agent = agente_analista
        )

In [14]:
# Cálculo determinista sobre qué señal de todas es la más fuerte. El llm a veces se confunde, se comparan.
import numpy as np

def balance_de_senales(registro: list) -> str:
    """Combina los lifts de todas las señales por producto (equivalente a acumular razones de
    verosimilitud asumiendo independencia). A diferencia de contar señales por dirección, esto
    respeta la magnitud: una señal de x4 no queda neutralizada por varias de x0.8."""
    lifts = [float(x) for llamada in registro
             for x in re.findall(r"x(\d+\.\d+)", llamada['resultado'])]
    if not lifts:
        return "riesgo_en_linea_con_la_media"
    combinado = float(np.prod(lifts))
    if combinado >= 1.20:
        return "riesgo_por_encima_de_la_media"
    if combinado <= 0.85:
        return "riesgo_por_debajo_de_la_media"
    return "riesgo_en_linea_con_la_media"

In [15]:
# Debido a los límites de tokens por hora y diarios de la API gratuita de Groq, hay que jugar con los tiempos
import time
import re
import asyncio

def extraer_espera_sugerida(error_msg: str, default: float = 65.0) -> float:
    match = re.search(r"try again in ([\d.]+)s", error_msg)
    sugerida = float(match.group(1)) + 5 if match else default
    return max(sugerida, 60.0)   # nunca menos de una ventana completa

async def ejecutar_con_reintentos(crew, max_intentos=6, espera_default=65):
    for intento in range(1, max_intentos + 1):
        try:
            return await crew.kickoff_async()
        except Exception as e:
            error_str = str(e)
            es_rate_limit = "RateLimitError" in str(type(e)) or "rate_limit" in error_str.lower()
            es_tool_corrupta = "tool_use_failed" in error_str or "<|channel|>" in error_str

            if es_rate_limit:
                espera = extraer_espera_sugerida(error_str, default=espera_default * intento)
                print(f"⚠️ Rate limit, esperando {espera:.1f}s (intento {intento}/{max_intentos})...")
            elif es_tool_corrupta:
                espera = 5
                print(f"⚠️ Nombre de tool corrupto, reintento inmediato "
                      f"({intento}/{max_intentos})...")
            else:
                raise
            await asyncio.sleep(espera)
    return None

### 6.5 Prueba con 1 fila del dataset (la misma que el predictor)

In [ ]:
# Cargar fila de prueba
X_test_raw = pd.read_csv('X_test_raw.csv')
fila = X_test_raw.iloc[8]
nombre_cliente = fila['Surname']
customer_data = fila.drop(labels=['RowNumber', 'CustomerId', 'Surname']).to_dict()

tarea_test = analizar_cliente(nombre_cliente, customer_data)

crew_analista = Crew(
    agents=[agente_analista],
    tasks=[tarea_test],
    process=Process.sequential
)

iniciar_registro()
resultado_analisis = await ejecutar_con_reintentos(crew_analista)

print(resultado_analisis)
print("\nTools realmente invocadas:", tools_llamadas())

uso = getattr(resultado_analisis, 'token_usage', None)
if uso:
    print(f"Tokens consumidos: {uso.total_tokens} "
          f"(prompt {uso.prompt_tokens} / respuesta {uso.completion_tokens}) — límite 8000/min")

analisis = resultado_analisis.pydantic
esperada = balance_de_senales(registro_llamadas)

print(f"Orientación del agente: {analisis.orientacion}")
print(f"Balance determinista:   {esperada}")
if analisis.orientacion != esperada:
    print("⚠️ No coinciden")
print(f"\n{analisis.conclusion}")

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Task: Analiza el perfil del cliente 'Lo' frente al histórico de la base.                                       │
│  Datos del cliente:                                                                                             │
│  - CreditScore: 505                                                                                             │
│  - Geography: France                                                                                            │
│  - Gender: Male                                                                                                 │
│  - Age: 49                                                                                                      │
│  - Tenure: 7                                                                                                    │
│  - Balance: 80001.23                                                                                            │
│  - NumOfProducts: 1                                                                                             │
│  - HasCrCard: 0                                                                                                 │
│  - IsActiveMember: 0                                                                                            │
│  - EstimatedSalary: 135180.11                                                                                   │
│                                                                                                                 │
│  Empieza por las variables de comportamiento del cliente con la entidad (productos contratados y estado de      │
│  actividad), que son las que más separan la base histórica. Añade después las variables de perfil (saldo y      │
│  contexto demográfico) solo si su resultado puede cambiar la lectura del caso. Responde en español.Al elegir    │
│  la orientación, pondera la intensidad de cada señal, no su número: una señal muy alejada de la media (por      │
│  ejemplo x4) pesa más que varias señales ligeramente por debajo. Una sola señal extrema puede determinar la     │
│  orientación.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool productos executed with result: Clientes con 1 producto(s) (n=4071): churn 28.0% vs 20.4% base → x1.37, por encima de la media...
Tool actividad executed with result: Clientes inactivos y sin tarjeta (n=1119): churn 25.7% vs 20.4% base → x1.26, por encima de la media...
Tool balance executed with result: Saldo de 80,001.23 € → percentil 42.1 de la base. Clientes del primer cuartil de saldo positivo (n=1278): churn 21.0% vs 20.4% base → x1.03, en línea con la media...


ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kwyzjma5f0js5544gtv6je92` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7291, Requested 2313. Please try again in 12.03s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



Tool contexto executed with result: [País] Clientes de France (n=3993): churn 15.9% vs 20.4% base → x0.78, por debajo de la media
[Edad] Clientes de 40-49 años (n=2079): churn 30.5% vs 20.4% base → x1.50, por encima de la media...


⚠️ Rate limit, esperando 60.0s (intento 1/6)...


...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Task: Analiza el perfil del cliente 'Lo' frente al histórico de la base.                                       │
│  Datos del cliente:                                                                                             │
│  - CreditScore: 505                                                                                             │
│  - Geography: France                                                                                            │
│  - Gender: Male                                                                                                 │
│  - Age: 49                                                                                                      │
│  - Tenure: 7                                                                                                    │
│  - Balance: 80001.23                                                                                            │
│  - NumOfProducts: 1                                                                                             │
│  - HasCrCard: 0                                                                                                 │
│  - IsActiveMember: 0                                                                                            │
│  - EstimatedSalary: 135180.11                                                                                   │
│                                                                                                                 │
│  Empieza por las variables de comportamiento del cliente con la entidad (productos contratados y estado de      │
│  actividad), que son las que más separan la base histórica. Añade después las variables de perfil (saldo y      │
│  contexto demográfico) solo si su resultado puede cambiar la lectura del caso. Responde en español.Al elegir    │
│  la orientación, pondera la intensidad de cada señal, no su número: una señal muy alejada de la media (por      │
│  ejemplo x4) pesa más que varias señales ligeramente por debajo. Una sola señal extrema puede determinar la     │
│  orientación.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool productos executed with result: Clientes con 1 producto(s) (n=4071): churn 28.0% vs 20.4% base → x1.37, por encima de la media...
Tool actividad executed with result: Clientes inactivos y sin tarjeta (n=1119): churn 25.7% vs 20.4% base → x1.26, por encima de la media...
Tool balance executed with result: Saldo de 80,001.23 € → percentil 42.1 de la base. Clientes del primer cuartil de saldo positivo (n=1278): churn 21.0% vs 20.4% base → x1.03, en línea con la media...
Tool contexto executed with result: [País] Clientes de France (n=3993): churn 15.9% vs 20.4% base → x0.78, por debajo de la media
[Edad] Clientes de 40-49 años (n=2079): churn 30.5% vs 20.4% base → x1.50, por encima de la media...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "cliente": "Lo",                                                                                             │
│    "senales": [                                                                                                 │
│      "productos: churn 28.0% vs 20.4% base → x1.37",                                                            │
│      "actividad: churn 25.7% vs 20.4% base → x1.26",                                                            │
│      "balance: churn 21.0% vs 20.4% base → x1.03",                                                              │
│      "contexto: churn 15.9% vs 20.4% base (France) → x0.78",                                                    │
│      "contexto: churn 30.5% vs 20.4% base (40-49 años) → x1.50"                                                 │
│    ],                                                                                                           │
│    "orientacion": "riesgo_por_encima_de_la_media",                                                              │
│    "conclusion": "El cliente presenta señales de mayor riesgo de abandono, especialmente por su bajo número de  │
│  productos, inactividad y pertenencia a la franja de edad 40-49, pese a un saldo cercano a la media y a la      │
│  baja tasa de churn en su país."                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

cliente='Lo' senales=['productos: churn 28.0% vs 20.4% base → x1.37', 'actividad: churn 25.7% vs 20.4% base → x1.26', 'balance: churn 21.0% vs 20.4% base → x1.03', 'contexto: churn 15.9% vs 20.4% base (France) → x0.78', 'contexto: churn 30.5% vs 20.4% base (40-49 años) → x1.50'] orientacion='riesgo_por_encima_de_la_media' conclusion='El cliente presenta señales de mayor riesgo de abandono, especialmente por su bajo número de productos, inactividad y pertenencia a la franja de edad 40-49, pese a un saldo cercano a la media y a la baja tasa de churn en su país.'

Tools realmente invocadas: ['Riesgo por número de productos', 'Score de actividad y abandono por segmento', 'Percentil de balance', 'Contexto demográfico']
Tokens consumidos: 32054 (prompt 28067 / respuesta 3987) — límite 8000/min
Orientación del agente: riesgo_por_encima_de_la_media
Balance determinista:   riesgo_por_encima_de_la_media

El cliente presenta señales de mayor riesgo de abandono, especialmente por su bajo número de

### 6.6 Tarea prueba Variabilidad

In [16]:
# 5 perfiles variados de test para estudiar

X_test_raw = pd.read_csv('X_test_raw.csv')


def primero(df_filtrado, fallback):
    """Devuelve la primera fila que cumpla el filtro, o el fallback si no hay ninguna."""
    return df_filtrado.iloc[0] if not df_filtrado.empty else fallback


perfiles_test = {
    # Saldo en el extremo bajo
    'balance_cero': primero(X_test_raw[X_test_raw['Balance'] == 0], X_test_raw.iloc[0]),

    # Saldo en el extremo alto
    'balance_alto': X_test_raw.sort_values('Balance', ascending=False).iloc[0],

    # Productos fuera de la zona estable
    'multi_producto': primero(X_test_raw[X_test_raw['NumOfProducts'] >= 3], X_test_raw.iloc[0]),

    # Eje de actividad y país de riesgo
    'inactivo_alemania': primero(
        X_test_raw[(X_test_raw['IsActiveMember'] == 0) & (X_test_raw['Geography'] == 'Germany')],
        X_test_raw.iloc[0]),

    # Sin nada llamativo en ningún eje: debería disparar pocas herramientas
    'perfil_limpio': primero(
        X_test_raw[(X_test_raw['IsActiveMember'] == 1)
                   & (X_test_raw['NumOfProducts'] == 2)
                   & (X_test_raw['Age'].between(28, 38))
                   & (X_test_raw['Geography'] != 'Germany')
                   & (X_test_raw['Balance'] > 0)],
        X_test_raw.iloc[0]),
}

for nombre, fila in perfiles_test.items():
    print(f"{nombre:18} -> {fila['Surname']:12} | Balance: {fila['Balance']:>10.0f} | "
          f"Prod: {fila['NumOfProducts']} | Activo: {fila['IsActiveMember']} | "
          f"{fila['Geography']} | {fila['Age']} años")

balance_cero       -> Hsiung       | Balance:          0 | Prod: 2 | Activo: 0 | France | 36 años
balance_alto       -> Lo           | Balance:     250898 | Prod: 3 | Activo: 1 | Spain | 38 años
multi_producto     -> Gardiner     | Balance:     118317 | Prod: 4 | Activo: 0 | France | 48 años
inactivo_alemania  -> O'Donnell    | Balance:     131024 | Prod: 2 | Activo: 0 | Germany | 33 años
perfil_limpio      -> Sun          | Balance:     116122 | Prod: 2 | Activo: 1 | France | 37 años


In [ ]:
N_REPETICIONES = 5
resultados_variabilidad = []

for perfil_nombre, fila in perfiles_test.items():
    nombre_cliente = fila['Surname']
    customer_data = fila.drop(labels=['RowNumber', 'CustomerId', 'Surname']).to_dict()

    for i in range(N_REPETICIONES):
        iniciar_registro()

        tarea = analizar_cliente(nombre_cliente, customer_data)
        crew = Crew(agents=[agente_analista], tasks=[tarea], process=Process.sequential)

        try:
            resultado = await ejecutar_con_reintentos(crew)
        except Exception as e:
            print(f"❌ [{perfil_nombre} - run {i+1}] Error no recuperable: {e}")
            resultado = None

        fila_res = {'perfil': perfil_nombre, 'cliente': nombre_cliente, 'run': i + 1}

        if resultado is None:
            print(f"❌ [{perfil_nombre} - run {i+1}] Fallido tras reintentos, se omite.")
            fila_res.update({'tools_llamadas': None, 'n_tools': None,
                             'orientacion': None, 'coincide_balance': None, 'tokens': None})
        else:
            tools = tools_llamadas()
            esperada = balance_de_senales(registro_llamadas)
            analisis = resultado.pydantic
            uso = getattr(resultado, 'token_usage', None)

            fila_res.update({
                'tools_orden': list(tools),          # orden real de llamada
                'tools_llamadas': sorted(tools),     # conjunto, para el Jaccard
                'n_tools': len(tools),
                'orientacion': analisis.orientacion if analisis else None,
                'coincide_balance': (analisis.orientacion == esperada) if analisis else None,
                'conclusion': analisis.conclusion if analisis else None,
                'tokens': uso.total_tokens if uso else None,
            })
            print(f"[{perfil_nombre} - run {i+1}] {len(tools)} tools: {sorted(tools)}")

        resultados_variabilidad.append(fila_res)
        pd.DataFrame(resultados_variabilidad).to_csv(
            'experimento_variabilidad_tools_5perfiles_gpt120b.csv', index=False)

        await asyncio.sleep(60)

df_variabilidad = pd.DataFrame(resultados_variabilidad)
df_variabilidad

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Task: Analiza el perfil del cliente 'Hsiung' frente al histórico de la base.                                   │
│  Datos del cliente:                                                                                             │
│  - CreditScore: 585                                                                                             │
│  - Geography: France                                                                                            │
│  - Gender: Male                                                                                                 │
│  - Age: 36                                                                                                      │
│  - Tenure: 7                                                                                                    │
│  - Balance: 0.0                                                                                                 │
│  - NumOfProducts: 2                                                                                             │
│  - HasCrCard: 1                                                                                                 │
│  - IsActiveMember: 0                                                                                            │
│  - EstimatedSalary: 94283.09                                                                                    │
│                                                                                                                 │
│  Empieza por las variables de comportamiento del cliente con la entidad (productos contratados y estado de      │
│  actividad), que son las que más separan la base histórica. Añade después las variables de perfil (saldo y      │
│  contexto demográfico) solo si su resultado puede cambiar la lectura del caso. Responde en español.Al elegir    │
│  la orientación, pondera la intensidad de cada señal, no su número: una señal muy alejada de la media (por      │
│  ejemplo x4) pesa más que varias señales ligeramente por debajo. Una sola señal extrema puede determinar la     │
│  orientación.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool productos executed with result: Clientes con 2 producto(s) (n=3672): churn 7.4% vs 20.4% base → x0.36, muy por debajo de la media...


ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: litellm.BadRequestError: GroqException - {"error":{"message":"Tool call validation failed: tool call validation failed: attempted to call tool 'actividad\u003c|channel|\u003ecommentary' which was not in request.tools","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"{\"name\": \"actividad\u003c|channel|\u003ecommentary\", \"arguments\": {\"has_crcard\":1,\"is_active_member\":0}}"}}



⚠️ Nombre de tool corrupto, reintento inmediato (1/6)...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Task: Analiza el perfil del cliente 'Hsiung' frente al histórico de la base.                                   │
│  Datos del cliente:                                                                                             │
│  - CreditScore: 585                                                                                             │
│  - Geography: France                                                                                            │
│  - Gender: Male                                                                                                 │
│  - Age: 36                                                                                                      │
│  - Tenure: 7                                                                                                    │
│  - Balance: 0.0                                                                                                 │
│  - NumOfProducts: 2                                                                                             │
│  - HasCrCard: 1                                                                                                 │
│  - IsActiveMember: 0                                                                                            │
│  - EstimatedSalary: 94283.09                                                                                    │
│                                                                                                                 │
│  Empieza por las variables de comportamiento del cliente con la entidad (productos contratados y estado de      │
│  actividad), que son las que más separan la base histórica. Añade después las variables de perfil (saldo y      │
│  contexto demográfico) solo si su resultado puede cambiar la lectura del caso. Responde en español.Al elegir    │
│  la orientación, pondera la intensidad de cada señal, no su número: una señal muy alejada de la media (por      │
│  ejemplo x4) pesa más que varias señales ligeramente por debajo. Una sola señal extrema puede determinar la     │
│  orientación.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool productos executed with result: Clientes con 2 producto(s) (n=3672): churn 7.4% vs 20.4% base → x0.36, muy por debajo de la media...


ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: litellm.BadRequestError: GroqException - {"error":{"message":"Tool call validation failed: tool call validation failed: attempted to call tool 'actividad\u003c|channel|\u003ecommentary' which was not in request.tools","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"{\"name\": \"actividad\u003c|channel|\u003ecommentary\", \"arguments\": {\"has_crcard\":1,\"is_active_member\":0}}"}}



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CancelledError: 

In [ ]:
# df_variabilidad = pd.read_csv('experimento_variabilidad_tools_3.csv')
# display(df_variabilidad)

In [ ]:
# La columna 'tokens' viene acumulada a lo largo de la sesión: la diferencia entre
# filas consecutivas es lo que costó realmente cada ejecución.
df_variabilidad['tokens_run'] = df_variabilidad['tokens'].diff()
df_variabilidad.loc[0, 'tokens_run'] = df_variabilidad.loc[0, 'tokens']  # la 1ª parte de cero

print(f"Coste por ejecución — media: {df_variabilidad['tokens_run'].mean():.0f} tokens, "
      f"mín: {df_variabilidad['tokens_run'].min():.0f}, "
      f"máx: {df_variabilidad['tokens_run'].max():.0f}")

print("\nMedia por perfil:")
print(df_variabilidad.groupby('perfil')['tokens_run'].mean().round(0))

Coste por ejecución — media: 10088 tokens, mín: 7450, máx: 23357

Media por perfil:
perfil
balance_alto          8636.0
balance_cero         12083.0
inactivo_alemania    12344.0
multi_producto        9785.0
perfil_limpio         7594.0
Name: tokens_run, dtype: float64


In [ ]:
from itertools import combinations
import numpy as np

def jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


print("=== Estabilidad por perfil (Jaccard medio entre pares de runs) ===")
for perfil_nombre in perfiles_test:
    subset = df_variabilidad[
        (df_variabilidad['perfil'] == perfil_nombre)
        & df_variabilidad['tools_llamadas'].notna()
    ]['tools_llamadas'].tolist()
    pares = list(combinations(subset, 2))
    if pares:
        print(f"{perfil_nombre}: {np.mean([jaccard(a, b) for a, b in pares]):.2f}")
    else:
        print(f"{perfil_nombre}: menos de 2 ejecuciones válidas, no se puede medir")

print("\n=== Discriminación entre perfiles (tools más frecuentes en cada uno) ===")
for perfil_nombre in perfiles_test:
    subset = df_variabilidad[
        (df_variabilidad['perfil'] == perfil_nombre)
        & df_variabilidad['tools_llamadas'].notna()
    ]
    if subset.empty:
        continue
    todas = [t for lista in subset['tools_llamadas'] for t in lista]
    n_runs = len(subset)
    print(f"\n{perfil_nombre} (media {subset['n_tools'].mean():.1f} tools):")
    for tool_nombre in sorted(set(todas)):
        print(f"   {todas.count(tool_nombre)}/{n_runs}  {tool_nombre}")

=== Estabilidad por perfil (Jaccard medio entre pares de runs) ===
balance_cero: 0.85
balance_alto: 1.00
multi_producto: 0.90
inactivo_alemania: 0.90
perfil_limpio: 1.00

=== Discriminación entre perfiles (tools más frecuentes en cada uno) ===

balance_cero (media 3.4 tools):
   2/5  Contexto demográfico
   5/5  Percentil de balance
   5/5  Riesgo por número de productos
   5/5  Score de actividad y abandono por segmento

balance_alto (media 3.0 tools):
   5/5  Percentil de balance
   5/5  Riesgo por número de productos
   5/5  Score de actividad y abandono por segmento

multi_producto (media 3.2 tools):
   1/5  Contexto demográfico
   5/5  Percentil de balance
   5/5  Riesgo por número de productos
   5/5  Score de actividad y abandono por segmento

inactivo_alemania (media 3.2 tools):
   1/5  Contexto demográfico
   5/5  Percentil de balance
   5/5  Riesgo por número de productos
   5/5  Score de actividad y abandono por segmento

perfil_limpio (media 3.0 tools):
   5/5  Percentil de

## 7. Agente Recomendador

La sig celda crea un json con las entradas al recomendador como si vinieran del analista y del predictor, pero sin venir de ellos realmente. Se hace para hacer pruebas y no consumir tokens todo el rato del analista y predictor.

In [17]:
perfiles_test['desacuerdo_franco'] = X_test_raw.loc[1198]  #añadimos otro perfil que de desacuerdo entre predictor y analista

In [18]:
import json
import re

def _invocar(tool_obj, **kwargs):
    """Llama a una tool de CrewAI directamente desde Python, sin pasar por el LLM."""
    for attr in ('run', 'func', '_run'):
        f = getattr(tool_obj, attr, None)
        if callable(f):
            return f(**kwargs)
    raise AttributeError(f"No se pudo invocar {tool_obj}")


def construir_caso(fila) -> dict:
    """Reproduce en Python lo que produciría el pipeline para un cliente: la predicción del
    modelo y las señales del analista. Determinista y sin coste, sirve como entrada fija para
    desarrollar el recomendador sin reejecutar predictor y analista en cada iteración."""
    customer_data = fila.drop(labels=['RowNumber', 'CustomerId', 'Surname']).to_dict()

    pred = predict_churn(customer_data)

    iniciar_registro()
    senales = [
        _invocar(consultar_riesgo_productos, num_productos=int(fila['NumOfProducts'])),
        _invocar(consultar_actividad_vinculacion,
                 is_active_member=int(fila['IsActiveMember']),
                 has_crcard=int(fila['HasCrCard'])),
        _invocar(calcular_percentil_balance, balance=float(fila['Balance'])),
        _invocar(consultar_contexto_demografico,
                 geography=fila['Geography'], age=int(fila['Age'])),
    ]

    return {
        'cliente': fila['Surname'],
        'datos': customer_data,
        'prediccion': {
            'churn_prediction': int(pred['churn_prediction']),
            'churn_probability_pct': round(pred['churn_probability'] * 100, 2),
        },
        'senales': senales,
        'orientacion_calculada': balance_de_senales(registro_llamadas),
    }


casos = {nombre: construir_caso(fila) for nombre, fila in perfiles_test.items()}

with open('casos_recomendador.json', 'w', encoding='utf-8') as f:
    json.dump(casos, f, ensure_ascii=False, indent=2)

for nombre, caso in casos.items():
    print(f"\n=== {nombre} — {caso['cliente']} ===")
    print(f"Predicción: {caso['prediccion']['churn_prediction']} "
          f"({caso['prediccion']['churn_probability_pct']}%)  |  "
          f"Orientación: {caso['orientacion_calculada']}")
    for s in caso['senales']:
        print(f"  - {s}")


=== balance_cero — Hsiung ===
Predicción: 0 (6.07%)  |  Orientación: riesgo_por_debajo_de_la_media
  - Clientes con 2 producto(s) (n=3672): churn 7.4% vs 20.4% base → x0.36, muy por debajo de la media
  - Clientes inactivos y con tarjeta (n=2762): churn 27.1% vs 20.4% base → x1.33, por encima de la media
  - Clientes con saldo 0 € (n=2891): churn 13.6% vs 20.4% base → x0.67, por debajo de la media
  - [País] Clientes de France (n=3993): churn 15.9% vs 20.4% base → x0.78, por debajo de la media
[Edad] Clientes de 30-39 años (n=3486): churn 10.9% vs 20.4% base → x0.54, muy por debajo de la media

=== balance_alto — Lo ===
Predicción: 1 (95.13%)  |  Orientación: riesgo_por_encima_de_la_media
  - Clientes con 3 producto(s) (n=207): churn 82.1% vs 20.4% base → x4.03, muy por encima de la media
  - Clientes activos y sin tarjeta (n=1211): churn 17.2% vs 20.4% base → x0.84, por debajo de la media
  - Saldo de 250,898.09 € → percentil 100.0 de la base. Clientes del cuarto cuartil de saldo pos

### 7.1 Router de tres rutas diferentes(decidido en Python, nunca por un LLM)

Combina la probabilidad del Predictor (riesgo oficial) con la agregación determinista del
Analista (`balance_de_senales`, nunca la `orientacion` que redacta el LLM ya que con temperature=0 y seed=42 la orientación del LLM
varía entre ejecuciones para el mismo cliente y lo importante son los valores que saca el analista, no cómo interpreta los datos para pensar en la posible orientacion de churn).

In [4]:
# Router de tres rutas
UMBRAL_PROB_ALTA = 50.0    # % de churn del Predictor a partir del cual el riesgo es "alto"
UMBRAL_PROB_BAJA = 30.0    # % por debajo del cual el Predictor lo da por "bajo"
UMBRAL_VALOR_ALTO = 75.0   # percentil de saldo a partir del cual se activa la variante "alto_valor"


def percentil_balance_numerico(balance: float) -> float:
    """Mismo cálculo que usa la tool 'Balance' del Analista, pero como número (no como
    texto), para decidir la variante de agente sin tener que parsear el resultado de la
    tool. Un saldo de 0 € se trata como percentil 0: es el extremo bajo de valor, no un
    caso especial de churn (eso ya lo cubre la propia tool del Analista)."""
    if balance <= 0:
        return 0.0
    return float((df_train['Balance'] < balance).mean() * 100)


def enrutar_cliente(churn_probability_pct: float, orientacion_calculada: str, balance: float) -> dict:
    """Decide la ruta de un cliente. Por diseño, la probabilidad del Predictor manda: si es
    alta, va a Ruta 2 aunque el Analista no esté de acuerdo, porque el Predictor es el
    riesgo oficial y el Analista solo aporta evidencia. Ruta 3 recoge todo lo que no es un cierre
    claro (Ruta 1) ni un riesgo claramente alto (Ruta 2): tanto el desacuerdo franco entre
    fuentes como la zona intermedia del Predictor caen ahí, porque en ambos casos falta
    certeza y conviene forzar el análisis adicional (simulación reanálisis) antes de
    recomendar nada."""
    riesgo_alto = churn_probability_pct >= UMBRAL_PROB_ALTA
    riesgo_bajo = churn_probability_pct < UMBRAL_PROB_BAJA
    analista_bajo = orientacion_calculada == "riesgo_por_debajo_de_la_media"

    percentil_valor = percentil_balance_numerico(balance)
    variante = "alto_valor" if percentil_valor >= UMBRAL_VALOR_ALTO else "cartera_masiva"

    if riesgo_bajo and analista_bajo:
        return {
            'ruta': 1, 'nombre_ruta': 'cierre_sin_recomendador', 'variante': None,
            'percentil_valor': percentil_valor,
            'motivo': (f"Predictor bajo ({churn_probability_pct:.1f}%) y Analista bajo "
                       f"({orientacion_calculada}): ambas fuentes coinciden en que no hay caso."),
        }

    if riesgo_alto:
        return {
            'ruta': 2, 'nombre_ruta': 'recomendador_riesgo_alto', 'variante': variante,
            'percentil_valor': percentil_valor,
            'motivo': (f"Predictor alto ({churn_probability_pct:.1f}%) → Recomendador, "
                       f"variante '{variante}' (percentil de saldo {percentil_valor:.1f})."),
        }

    #si riesgo bajo y analista alto
    return {
        'ruta': 3, 'nombre_ruta': 'recomendador_revision', 'variante': variante,
        'percentil_valor': percentil_valor,
        'motivo': (f"Predictor ({churn_probability_pct:.1f}%) no llega a riesgo alto y no hay "
                   f"acuerdo claro de riesgo bajo con el Analista ({orientacion_calculada}): "
                   f"Recomendador en modo revisión, con simulación reanálisis obligatoria."),
    }

### 7.2 Reparto de rutas sobre los 5 casos congelados

Aplica el router a los `casos` del json. Comprobar si los umbrales provisionales de 7.1 dejan a algún
cliente justo en el borde antes de darlos por definitivos.

In [20]:
# Reparto de rutas sobre los 5 casos congelados
def construir_reparto(casos: dict) -> pd.DataFrame:
    """Aplica el router a los casos ya calculados y devuelve la tabla de reparto: qué ruta y
    qué variante le tocó a cada cliente, y por qué."""
    filas = []
    for perfil, caso in casos.items():
        info = enrutar_cliente(
            churn_probability_pct=caso['prediccion']['churn_probability_pct'],
            orientacion_calculada=caso['orientacion_calculada'],
            balance=caso['datos']['Balance'],
        )
        filas.append({
            'perfil': perfil,
            'cliente': caso['cliente'],
            'probabilidad_pct': caso['prediccion']['churn_probability_pct'],
            'orientacion_analista': caso['orientacion_calculada'],
            'percentil_saldo': round(info['percentil_valor'], 1),
            'ruta': info['nombre_ruta'],
            'variante': info['variante'],
            'motivo': info['motivo'],
        })
    return pd.DataFrame(filas)


reparto = construir_reparto(casos)
pd.set_option('display.max_colwidth', 120)
display(reparto)

print("\nDistribución de rutas:")
print(reparto['ruta'].value_counts())

,perfil,cliente,probabilidad_pct,orientacion_analista,percentil_saldo,ruta,variante,motivo
0,balance_cero,Hsiung,6.07,riesgo_por_debajo_de_la_media,0.0,cierre_sin_recomendador,None,Predictor bajo (6.1%) y Analista bajo (riesgo_por_debajo_de_la_media): ambas fuentes coinciden en que no hay caso.
1,balance_alto,Lo,95.13,riesgo_por_encima_de_la_media,100.0,recomendador_riesgo_alto,alto_valor,"Predictor alto (95.1%) → Recomendador, variante 'alto_valor' (percentil de saldo 100.0)."
2,multi_producto,Gardiner,97.59,riesgo_por_encima_de_la_media,66.9,recomendador_riesgo_alto,cartera_masiva,"Predictor alto (97.6%) → Recomendador, variante 'cartera_masiva' (percentil de saldo 66.9)."
3,inactivo_alemania,O'Donnell,19.70,riesgo_por_debajo_de_la_media,78.0,cierre_sin_recomendador,None,Predictor bajo (19.7%) y Analista bajo (riesgo_por_debajo_de_la_media): ambas fuentes coinciden en que no hay caso.
4,perfil_limpio,Sun,32.83,riesgo_por_debajo_de_la_media,65.1,recomendador_revision,cartera_masiva,Predictor (32.8%) no llega a riesgo alto y no hay acuerdo claro de riesgo bajo con el Analista (riesgo_por_debajo_de...
5,desacuerdo_franco,Gray,17.37,riesgo_por_encima_de_la_media,41.6,recomendador_revision,cartera_masiva,Predictor (17.4%) no llega a riesgo alto y no hay acuerdo claro de riesgo bajo con el Analista (riesgo_por_encima_de...



Distribución de rutas:
ruta
cierre_sin_recomendador     2
recomendador_riesgo_alto    2
recomendador_revision       2
Name: count, dtype: int64


### 7.3 Tool de simulación re-análisis

Re-invoca `predict_churn` cambiando, una a una, las variables sobre las que el banco puede
intervenir de verdad (activación, tarjeta, número de productos) y devuelve solo el delta en
puntos porcentuales frente a la situación actual del cliente. solo la usa cuando
el caso lo justifica: riesgo alto (Ruta 2, opcional) o desacuerdo entre fuentes (Ruta 3,
obligatoria).

In [21]:
# Tool de simulación re-analisis
@tool("Simulación reanalisis")
def simular_reanalisis_tool(
    CreditScore: int, Geography: str, Gender: str, Age: int, Tenure: int,
    Balance: float, NumOfProducts: int, HasCrCard: int,
    IsActiveMember: int, EstimatedSalary: float
) -> str:
    """
    ÚTIL ÚNICAMENTE PARA: estimar qué palanca de retención movería más la probabilidad de
    churn de ESTE cliente concreto. Úsala cuando el riesgo sea alto o cuando el Predictor y
    el Analista no coincidan: en ambos casos conviene saber sobre qué variable actuar antes
    de proponer nada. No la uses con un cliente de riesgo bajo.
    Re-calcula la probabilidad del modelo cambiando, una a una, las variables sobre las que
    el banco puede intervenir realmente (activar al cliente, ofrecerle tarjeta, o llevarlo
    al número de productos más estable). Devuelve SOLO el cambio en puntos porcentuales de
    cada intervención frente a la situación actual del cliente, de mayor a menor impacto —
    nunca la probabilidad absoluta, que es competencia exclusiva del Predictor.
    """
    base = {
        'CreditScore': CreditScore, 'Geography': Geography, 'Gender': Gender,
        'Age': Age, 'Tenure': Tenure, 'Balance': Balance,
        'NumOfProducts': NumOfProducts, 'HasCrCard': HasCrCard,
        'IsActiveMember': IsActiveMember, 'EstimatedSalary': EstimatedSalary,
    }
    try:
        prob_base = predict_churn(base)['churn_probability']
    except Exception as e:
        return f"Error al calcular la situación actual del cliente: {e}"

    candidatas = {}
    if IsActiveMember == 0:
        candidatas['Activar al cliente'] = {**base, 'IsActiveMember': 1}
    if HasCrCard == 0:
        candidatas['Ofrecer tarjeta de crédito'] = {**base, 'HasCrCard': 1}
    if NumOfProducts != _prod_estable:
        candidatas[f'Llevar a {_prod_estable} producto(s)'] = {**base, 'NumOfProducts': _prod_estable}

    if not candidatas:
        return ("Este cliente ya está en la posición más favorable en las tres variables "
                "sobre las que el banco puede intervenir (activo, con tarjeta, en el número "
                "de productos más estable). No hay palanca que simular; la recomendación "
                "tendrá que apoyarse solo en las señales del Analista.")

    resultados = []
    for nombre, modificado in candidatas.items():
        try:
            prob_mod = predict_churn(modificado)['churn_probability']
            resultados.append((nombre, (prob_mod - prob_base) * 100))
        except Exception:
            resultados.append((nombre, None))

    resultados.sort(key=lambda par: (par[1] is None, par[1]))
    lineas = [
        f"- {nombre}: {delta:+.1f} puntos porcentuales" if delta is not None
        else f"- {nombre}: error al simular"
        for nombre, delta in resultados
    ]
    resultado_texto = (
        "Impacto simulado de cada palanca disponible sobre la probabilidad de churn de "
        "este cliente, ordenado de mayor a menor reducción de riesgo (sensibilidad del "
        "modelo, no causalidad):\n" + "\n".join(lineas)
    )
    print("\n[Simulación y Reanálisis — resultado completo]")
    print(resultado_texto, "\n")
    return resultado_texto

### 7.4 Definición del agente (variante según valor del cliente)

Misma tarea, dos variantes de agente: cartera de alto valor (percentil de saldo por encima del
umbral) con trato de gestor dedicado, y cartera masiva con acciones escalables.

In [22]:
# Definición del agente — variante según valor del cliente
llm_recomendador = LLM(
    model="groq/openai/gpt-oss-20b",
    temperature=0.2,
    seed=42,
)

_BACKSTORY_COMUN = (
    'El nivel de riesgo oficial es la probabilidad que calculó el Predictor: no la '
    'cuestionas ni la vuelves a estimar. El Analista no emite un segundo veredicto de '
    'riesgo, solo evidencia (señales con sus cifras) sobre POR QUÉ; usa esas señales para '
    'decidir sobre qué palanca actuar, nunca para corregir el riesgo del Predictor.\n\n'
    'No inventas cifras: cualquier número que cites en tu justificación tiene que venir '
    'de las señales del Analista o de la tool de simulación reanálisis, si la usas. '
    'Siempre declaras una limitación: sensibilidad no causal si usaste la simulación, o '
    'que te apoyas en tasas históricas de segmentos similares (no en este cliente en '
    'particular) si no la usaste.\n\n'
    'Propones UNA acción de retención concreta, no una lista de opciones. Respondes en '
    'español.'
)


def crear_agente_recomendador(variante: str) -> Agent:
    """variante: 'alto_valor' o 'cartera_masiva'. Dos agentes con el mismo objetivo y las
    mismas herramientas, pero con un backstory distinto."""
    if variante == 'alto_valor':
        enfoque = (
            'Trabajas la cartera de alto valor: clientes en el cuartil alto de saldo. Cada acción la '
            'ejecuta y vigila el gestor personal asignado a ESTE cliente concreto — nunca una campaña '
            'automática ni una oferta genérica. Redacta "accion_principal" y "justificacion" dando voz '
            'al gestor y transmitiendo seguimiento individual (por ejemplo: "el gestor contacta al '
            'cliente para..." o "el gestor propone y hace seguimiento de..."), no como una oferta '
            'estándar aplicable a cualquiera. El canal es siempre "gestor personal".'
        )
    elif variante == 'cartera_masiva':
        enfoque = (
            'Trabajas la cartera masiva: volumen alto, trato estandarizado. Cada acción se aplica igual '
            'a cualquier cliente con un perfil similar — nunca una gestión personalizada de alto coste ni '
            'un seguimiento individual. Redacta "accion_principal" y "justificacion" como una acción '
            'escalable y general (por ejemplo: "se incluye en campaña automatizada de..." o "contact '
            'center ofrece..."), sin dar a entender trato ni seguimiento individualizado. El canal es '
            '"campaña automatizada" o "contact center", según la acción.'
        )
    else:
        raise ValueError(f"Variante de Recomendador no reconocida: {variante}")

    return Agent(
        role=f'Agente Recomendador de Retención — {variante}',
        goal='Proponer una acción de retención concreta y justificada para un cliente con '
             'riesgo de churn, apoyada en las señales del Analista y, cuando el caso lo '
             'requiera, en la simulación reanálisis.',
        backstory=f'{enfoque}\n\n{_BACKSTORY_COMUN}',
        tools=[simular_reanalisis_tool],
        verbose=True,
        llm=llm_recomendador,
    )


agente_recomendador_alto_valor = crear_agente_recomendador('alto_valor')
agente_recomendador_cartera_masiva = crear_agente_recomendador('cartera_masiva')

### 7.5 Salida estructurada y factoría de tareas

Los campos `ruta` y `variante_agente` no los decide el LLM: ya vienen fijados por el router de
7.1, y la tarea le pide que los rellene tal cual se le indican. La instrucción de usar la tool de reanalisis es
opcional en Ruta 2 y obligatoria en Ruta 3 (aquí donde vive el "solicitar nuevos análisis
cuando falte información)".

In [ ]:
# Salida estructurada y factoría de tareas
from typing import Optional

class RecomendacionChurn(BaseModel):
    cliente: str
    ruta: Literal["riesgo_alto", "revision"]
    variante_agente: Literal["alto_valor", "cartera_masiva"]
    resumen_cliente: str = Field(..., description="1-2 frases en lenguaje natural describiendo al cliente a partir "
                                  "de sus datos (edad, antigüedad, saldo, productos...), sin mencionar el riesgo.")
    resumen_riesgo: str = Field(..., description="1-2 frases: probabilidad del Predictor y las señales del Analista "
                                 "cuyo multiplicador (x) sea mayor que 1, que reespaldan el churn. No reevalúa el riesgo, solo lo resume.")
    accion_principal: str = Field(..., description="Acción de retención concreta y viable. Debe actuar sobre la palanca objetivo, "
                                    "pero no debe asumir que una modificación de la variable causa una reducción del churn "
                                    "ni ordenar directamente al cliente modificar sus productos, salvo que exista una "
                                    "justificación explícita independiente de la simulación.")
    palanca_objetivo: str = Field(..., description="Variable sobre la que actúa: productos, actividad, tarjeta u otra")
    canal: str = Field(..., description="Cómo se ejecuta: gestor personal, campaña automatizada, contact center, etc.")
    justificacion: str = Field(..., description="1-3 frases que expliquen por qué se propone la acción, apoyándose en las señales "
                                    "del Analista y, si se utilizó, en la simulación. El resultado de la simulación debe "
                                    "interpretarse como sensibilidad del modelo, no como evidencia de causalidad.")
    limitaciones: str = Field(..., description="Advertencia obligatoria (nunca vacío): sensibilidad no causal si se "
                               "usó la simulación reanálisis, o que la recomendación se basa en tasas históricas "
                               "de segmentos similares (no en una estimación para este cliente) si no se usó.")
def crear_tarea_recomendacion(
    nombre_cliente: str,
    customer_data: dict,
    prediccion: dict,
    senales: list,
    ruta_info: dict,
    agente: Agent,
    conclusion_analista: Optional[str] = None,
) -> Task:
    datos_str = ", ".join(f"{k}: {v}" for k, v in customer_data.items())
    senales_str = "\n".join(f"- {s}" for s in senales)
    bloque_conclusion = (f"\nConclusión redactada por el Analista: {conclusion_analista}\n"
                         if conclusion_analista else "")

    if ruta_info['ruta'] == 2 and ruta_info['variante'] == 'alto_valor':
        instruccion_ruta = (
            "Ruta: RIESGO ALTO, cartera de alto valor. El Predictor ya marca a este cliente "
            "como riesgo alto: no hace falta que justifiques el riesgo, solo la acción. DEBES "
            "invocar la tool de simulación reanálisis antes de proponer nada: en la cartera "
            "de alto valor la acción se justifica con una cifra concreta de impacto sobre ESTE "
            "cliente, no solo con las tasas del histórico. Prioriza la palanca con mayor delta, "
            "no necesariamente la primera que parezca plausible."
        )
        ruta_pydantic = "riesgo_alto"
    elif ruta_info['ruta'] == 2:
        instruccion_ruta = (
            "Ruta: RIESGO ALTO. El Predictor ya marca a este cliente como riesgo alto: no "
            "hace falta que justifiques el riesgo, solo la acción. Apóyate primero en las "
            "señales del Analista para decidir la palanca; usa la tool de simulación "
            "reanálisis solo si las señales no bastan para elegir entre dos palancas "
            "igual de plausibles."
        )
        ruta_pydantic = "riesgo_alto"
    else:
        instruccion_ruta = (
            "Ruta: REVISIÓN. El Predictor y la agregación determinista del Analista no "
            "coinciden con claridad. DEBES invocar la tool de simulación reanálisis antes"
            "de proponer nada: permite identificar qué palanca presenta una mayor "
            "sensibilidad en la predicción del modelo para ESTE cliente, "
            "en lugar de basarse únicamente en las tasas históricas del segmento."
        )
        ruta_pydantic = "revision"
    if ruta_info['variante'] == 'alto_valor':
            instruccion_estilo = (
                "Recuerda: eres el gestor personal de este cliente. Escribe 'accion_principal' y "
                "'justificacion' en ese papel — seguimiento individual, trato a medida — no como una "
                "oferta estándar aplicable a cualquiera."
            )
    else:
            instruccion_estilo = (
                "Recuerda: esta acción se aplica igual a cualquier cliente con perfil similar. Escribe "
                "'accion_principal' y 'justificacion' como una acción escalable y general, sin dar a "
                "entender trato o seguimiento individualizado."
            )
    return Task(
                description=(
            f"Cliente: {nombre_cliente}.\n"
            f"Datos del cliente: {datos_str}\n\n"
            f"Predicción del Predictor (riesgo oficial): {prediccion['churn_prediction']} "
            f"(0=no abandona, 1=abandona), probabilidad {prediccion['churn_probability_pct']:.2f}%.\n\n"
            f"Señales del Analista (evidencia, no un segundo veredicto de riesgo):\n{senales_str}\n"
            f"{bloque_conclusion}\n"
            f"{instruccion_ruta}\n\n"
            f"Motivo de esta ruta: {ruta_info['motivo']}\n\n"
            f"Rellena 'ruta' con exactamente '{ruta_pydantic}' y 'variante_agente' con "
            f"exactamente '{ruta_info['variante']}'; no los decidas tú, ya vienen fijados "
            f"por el enrutado en Python.\n\n"
            "En 'resumen_cliente', describe en 1-2 frases y en lenguaje natural quién es este cliente a "
            "partir de sus datos (edad, antigüedad, saldo, productos, etc.), sin mencionar el riesgo ni "
            "la acción — solo contexto.\n\n"
            "En 'resumen_riesgo', cita la probabilidad del Predictor y enumera TODAS las señales con "
            "multiplicador ('x') mayor que 1, no solo la más alta; ignora las menores que 1. Si ninguna "
            "supera 1, dilo explícitamente. No reevalúes el riesgo, solo sintetiza lo ya calculado.\n\n"
            "En 'limitaciones', nunca la dejes vacía: si usaste la tool reanálisis, indica que mide "
            "sensibilidad, no causalidad; si no la usaste, que la recomendación se basa en tasas "
            "históricas del segmento, no en una estimación individual de este cliente.\n\n"
            "Si una señal >1 de 'resumen_riesgo' no es una palanca intervenible (edad, país, saldo, etc.), "
            "menciónala en 'justificacion', pero en una frase aparte de las palancas intervenibles y con otro "
            "verbo: usa 'el caso se ve agravado por...' o 'además, el cliente presenta...', nunca 'la campaña "
            "aborda' ni 'la acción se dirige a' seguido de una señal no intervenible — ni siquiera si esa "
            "misma frase también menciona señales intervenibles."
            "Si en cambio una señal SÍ es intervenible pero la descartas como palanca, explica en "
            "'justificacion' por qué (ej. muestra pequeña). No ignores ninguna señal en silencio.\n\n"
            f"{instruccion_estilo}\n\n"
            "Propón una acción de retención concreta y viable para este cliente, dirigida a la palanca o "
            "palancas identificadas. No interpretes el resultado de la simulación como causalidad ni des "
            "por hecho que el cliente debe modificar esa variable: solo indica qué cambio reduciría más la "
            "probabilidad de churn según el modelo, no que vaya a evitarlo.\n\n"
            "Formula la acción como seguimiento del gestor, nunca como una orden al cliente. Para "
            "'NumOfProducts', no propongas cerrar ni cancelar un producto: propón que se revise con el "
            "cliente su cartera, se detecten productos infrautilizados o necesidades de consolidación, y se "
            "valoren ajustes junto con él, la decisión final es suya. Para 'HasCrCard', orienta la acción a "
            "revisar sus necesidades de tarjeta y ofrecer una solución adecuada. Para 'IsActiveMember', "
            "orienta la acción a revisar su nivel de uso de los servicios y, si procede, fomentarlo.\n\n"
            "Si más de una de las tres palancas intervenibles (activación, tarjeta, productos) tiene señal "
            "relevante para este cliente, combínalas en una única acción de seguimiento que cubra todas, no "
            "elijas solo una y descartes el resto. 'palanca_objetivo' debe nombrar todas las palancas que "
            "uses. Si citas el impacto simulado de varias palancas, dalo por separado, nunca sumado, la "
            "simulación no prueba combinaciones."
            ),
        expected_output=(
            "Un objeto con la acción de retención propuesta, la palanca a la que apunta, el "
            "canal de ejecución, la justificación apoyada en las señales citadas y las "
            "limitaciones a declarar."
        ),
        agent=agente,
        output_pydantic=RecomendacionChurn,
    )

### 7.6 Prueba end-to-end sobre un cliente

Cambia `casos['balance_alto']` por cualquier otra
clave de `casos` para probar otra ruta.

In [ ]:
import json, os

if os.path.exists('resultados_recomendador.json'):
    with open('resultados_recomendador.json', encoding='utf-8') as f:
        resultados_recomendador = json.load(f)
else:
    resultados_recomendador = {}

In [ ]:
# Prueba end-to-end sobre un cliente
perfil_prueba = 'desacuerdo_franco'   # ir cambiando este perfil
caso_prueba = casos[perfil_prueba]

ruta_info = enrutar_cliente(
    churn_probability_pct=caso_prueba['prediccion']['churn_probability_pct'],
    orientacion_calculada=caso_prueba['orientacion_calculada'],
    balance=caso_prueba['datos']['Balance'],
)
print(f"Ruta: {ruta_info}")

if ruta_info['ruta'] == 1:
    print("\nEste caso se cierra sin Recomendador según el router. "
          "Elige otra clave de 'casos' para probar el agente.")
else:
    agente_elegido = (agente_recomendador_alto_valor if ruta_info['variante'] == 'alto_valor'
                       else agente_recomendador_cartera_masiva)

    tarea_recomendacion = crear_tarea_recomendacion(
        nombre_cliente=caso_prueba['cliente'],
        customer_data=caso_prueba['datos'],
        prediccion=caso_prueba['prediccion'],
        senales=caso_prueba['senales'],
        ruta_info=ruta_info,
        agente=agente_elegido,
    )
    crew_recomendador = Crew(
        agents=[agente_elegido],
        tasks=[tarea_recomendacion],
        process=Process.sequential,
    )

    resultado_recomendacion = await ejecutar_con_reintentos(crew_recomendador)
    print(resultado_recomendacion)

    resultados_recomendador[perfil_prueba] = resultado_recomendacion.pydantic.model_dump()
    with open('resultados_recomendador.json', 'w', encoding='utf-8') as f:
        json.dump(resultados_recomendador, f, ensure_ascii=False, indent=2)
    print(f"\nGuardado en resultados_recomendador.json (clave: '{perfil_prueba}')")

Ruta: {'ruta': 3, 'nombre_ruta': 'recomendador_revision', 'variante': 'cartera_masiva', 'percentil_valor': 41.6375, 'motivo': 'Predictor (17.4%) no llega a riesgo alto y no hay acuerdo claro de riesgo bajo con el Analista (riesgo_por_encima_de_la_media): Recomendador en modo revisión, con simulación reanálisis obligatoria.'}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Recomendador de Retención — cartera_masiva                                                       │
│                                                                                                                 │
│  Task: Cliente: Gray.                                                                                           │
│  Datos del cliente: CreditScore: 613, Geography: Germany, Gender: Male, Age: 69, Tenure: 9, Balance: 78778.49,  │
│  NumOfProducts: 1, HasCrCard: 0, IsActiveMember: 1, EstimatedSalary: 8751.59                                    │
│                                                                                                                 │
│  Predicción del Predictor (riesgo oficial): 0 (0=no abandona, 1=abandona), probabilidad 17.37%.                 │
│                                                                                                                 │
│  Señales del Analista (evidencia, no un segundo veredicto de riesgo):                                           │
│  - Clientes con 1 producto(s) (n=4071): churn 28.0% vs 20.4% base → x1.37, por encima de la media               │
│  - Clientes activos y sin tarjeta (n=1211): churn 17.2% vs 20.4% base → x0.84, por debajo de la media           │
│  - Saldo de 78,778.49 € → percentil 41.6 de la base. Clientes del primer cuartil de saldo positivo (n=1278):    │
│  churn 21.0% vs 20.4% base → x1.03, en línea con la media                                                       │
│  - [País] Clientes de Germany (n=2005): churn 33.1% vs 20.4% base → x1.62, muy por encima de la media           │
│  [Edad] Clientes de 60-69 años (n=300): churn 36.0% vs 20.4% base → x1.77, muy por encima de la media           │
│                                                                                                                 │
│  Ruta: REVISIÓN. El Predictor y la agregación determinista del Analista no coinciden con claridad. DEBES        │
│  invocar la tool de simulación reanálisis antesde proponer nada: permite identificar qué palanca presenta una   │
│  mayor sensibilidad en la predicción del modelo para ESTE cliente, en lugar de basarse únicamente en las tasas  │
│  históricas del segmento.                                                                                       │
│                                                                                                                 │
│  Motivo de esta ruta: Predictor (17.4%) no llega a riesgo alto y no hay acuerdo claro de riesgo bajo con el     │
│  Analista (riesgo_por_encima_de_la_media): Recomendador en modo revisión, con simulación reanálisis             │
│  obligatoria.                                                                                                   │
│                                                                                                                 │
│  Rellena 'ruta' con exactamente 'revision' y 'variante_agente' con exactamente 'cartera_masiva'; no los         │
│  decidas tú, ya vienen fijados por el enrutado en Python.                                                       │
│                                                                                                                 │
│  En 'resumen_cliente', describe en 1-2 frases y en lenguaje natural quién es este cliente a partir de sus       │
│  datos (edad, antigüedad, saldo, productos, etc.), sin mencionar el riesgo ni la acción — solo contexto.        │
│                                                                                                                 │
│  En 'resumen_riesgo', cita la probabilidad del Predictor y enumera TODAS las señales con multiplicador ('x')    │
│  mayor que 1, no solo la más alta; ignora las menores q


[Simulación y Reanálisis — resultado completo]
Impacto simulado de cada palanca disponible sobre la probabilidad de churn de este cliente, ordenado de mayor a menor reducción de riesgo (sensibilidad del modelo, no causalidad):
- Llevar a 2 producto(s): -2.3 puntos porcentuales
- Ofrecer tarjeta de crédito: -0.7 puntos porcentuales 

Tool simulacion_reanalisis executed with result: Impacto simulado de cada palanca disponible sobre la probabilidad de churn de este cliente, ordenado de mayor a menor reducción de riesgo (sensibilidad del modelo, no causalidad):
- Llevar a 2 product...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Recomendador de Retención — cartera_masiva                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "cliente": "Gray",                                                                                           │
│    "ruta": "revision",                                                                                          │
│    "variante_agente": "cartera_masiva",                                                                         │
│    "resumen_cliente": "Cliente de 69 años, con 9 años de antigüedad en la entidad, saldo de 78.78 k €, un solo  │
│  producto y sin tarjeta de crédito.",                                                                           │
│    "resumen_riesgo": "El Predictor indica una probabilidad de churn del 17.37 %. Señales con multiplicador      │
│  mayor a 1: clientes con 1 producto (x1.37), clientes de Alemania (x1.62), clientes de 60‑69 años (x1.77).",    │
│    "accion_principal": "Se incluye en campaña automatizada de revisión de cartera para clientes con 1           │
│  producto, ofreciendo la posibilidad de añadir un segundo producto y revisar su cartera.",                      │
│    "palanca_objetivo": "NumOfProducts",                                                                         │
│    "canal": "campaña automatizada",                                                                             │
│    "justificacion": "La simulación reanálisis indica que añadir un segundo producto reduce la probabilidad de   │
│  churn en 2.3 pp, la mayor sensibilidad entre las palancas disponibles. Además, el caso se ve agravado por el   │
│  hecho de que el cliente es de Alemania (x1.62) y pertenece al grupo de edad 60‑69 años (x1.77).",              │
│    "limitaciones": "La recomendación se basa en la sensibilidad del modelo obtenida mediante la simulación      │
│  reanálisis; no implica causalidad."                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

cliente='Gray' ruta='revision' variante_agente='cartera_masiva' resumen_cliente='Cliente de 69 años, con 9 años de antigüedad en la entidad, saldo de 78.78\u202fk\u202f€, un solo producto y sin tarjeta de crédito.' resumen_riesgo='El Predictor indica una probabilidad de churn del 17.37\u202f%. Señales con multiplicador mayor a 1: clientes con 1 producto (x1.37), clientes de Alemania (x1.62), clientes de 60‑69 años (x1.77).' accion_principal='Se incluye en campaña automatizada de revisión de cartera para clientes con 1 producto, ofreciendo la posibilidad de añadir un segundo producto y revisar su cartera.' palanca_objetivo='NumOfProducts' canal='campaña automatizada' justificacion='La simulación reanálisis indica que añadir un segundo producto reduce la probabilidad de churn en 2.3\u202fpp, la mayor sensibilidad entre las palancas disponibles. Además, el caso se ve agravado por el hecho de que el cliente es de Alemania (x1.62) y pertenece al grupo de edad 60‑69 años (x1.77).' limitacion

## Sistema multiagente completo

In [ ]:
#Pipeline completo real: Predictor + Analista (en "paralelo") + Recomendador, por cliente.
import json, os

RUTA_RESULTADOS_PIPELINE = 'resultados_pipeline_completo.json'

if os.path.exists(RUTA_RESULTADOS_PIPELINE):
    with open(RUTA_RESULTADOS_PIPELINE, encoding='utf-8') as f:
        resultados_pipeline = json.load(f)
else:
    resultados_pipeline = {}


async def ejecutar_pipeline_cliente(perfil: str, forzar: bool = True) -> dict:
    """Ejecuta Predictor, Analista y, si el router lo requiere, el Recomendador, sobre un único
    perfil de 'casos'. Cada agente corre en su propio Crew de una sola tarea, en secuencia (el
    tier gratuito no permite paralelismo real ni resumir una tarea a medias — ver documento de
    estado). Si el perfil ya está completo, no gasta cuota (salvo forzar=True). Si ya tiene
    Predictor+Analista pero le falta el Recomendador (p.ej. porque agotó los reintentos la vez
    anterior), reintenta SOLO el Recomendador, reutilizando lo ya guardado."""

    registro_previo = resultados_pipeline.get(perfil)
    ya_completo = registro_previo is not None and (
        registro_previo['ruta_info']['ruta'] == 1 or registro_previo.get('recomendacion') is not None
    )

    if registro_previo is not None and ya_completo and not forzar:
        print(f"'{perfil}' ya está completo en {RUTA_RESULTADOS_PIPELINE}, no se llama a Groq "
              f"(usa forzar=True para reejecutar todo desde cero).")
        return registro_previo

    if registro_previo is not None and not ya_completo and not forzar:
        print(f"'{perfil}' ya tiene Predictor y Analista guardados; solo falta el Recomendador. "
              f"Reintentando solo esa etapa (no se repite Predictor ni Analista).")
        registro = registro_previo
        nombre_cliente = registro['cliente']
        customer_data = registro['datos']
        ruta_info = registro['ruta_info']
    else:
        caso = casos[perfil]
        nombre_cliente = caso['cliente']
        customer_data = caso['datos']

        # --- Etapa 1: Predictor ---
        tarea_pred = crear_tarea_prediccion(nombre_cliente, customer_data)
        crew_predictor = Crew(agents=[agente_predictor], tasks=[tarea_pred], process=Process.sequential)
        resultado_pred = await ejecutar_con_reintentos(crew_predictor)
        if resultado_pred is None:
            print(f"⚠️ '{perfil}': Predictor no completó tras los reintentos. Nada guardado.")
            return None
        prediccion_llm = resultado_pred.pydantic
        uso_pred = getattr(resultado_pred, 'token_usage', None)

        # --- Etapa 2: Analista ---
        tarea_analisis = analizar_cliente(nombre_cliente, customer_data)
        crew_analista = Crew(agents=[agente_analista], tasks=[tarea_analisis], process=Process.sequential)
        iniciar_registro()
        resultado_analisis = await ejecutar_con_reintentos(crew_analista)
        if resultado_analisis is None:
            print(f"⚠️ '{perfil}': Analista no completó tras los reintentos. Nada guardado "
                  f"(el coste del Predictor ya se ha gastado, pero no se guarda un caso a medias).")
            return None
        analisis_llm = resultado_analisis.pydantic
        orientacion_calculada = balance_de_senales(registro_llamadas)
        uso_analisis = getattr(resultado_analisis, 'token_usage', None)

        ruta_info = enrutar_cliente(
            churn_probability_pct=prediccion_llm.churn_probability_pct,
            orientacion_calculada=orientacion_calculada,
            balance=customer_data['Balance'],
        )

        registro = {
            'cliente': nombre_cliente,
            'datos': customer_data,
            'prediccion': prediccion_llm.model_dump(),
            'senales': analisis_llm.senales,
            'conclusion_analista': analisis_llm.conclusion,
            'orientacion_llm': analisis_llm.orientacion,
            'orientacion_calculada': orientacion_calculada,
            'ruta_info': ruta_info,
            'tokens_predictor': uso_pred.total_tokens if uso_pred else None,
            'tokens_analista': uso_analisis.total_tokens if uso_analisis else None,
            'recomendacion': None,
            'tokens_recomendador': None,
        }

        print(f"[{perfil}] {nombre_cliente} — Predictor {prediccion_llm.churn_probability_pct:.2f}% | "
              f"Analista: {orientacion_calculada} | Ruta: {ruta_info['nombre_ruta']}")

        # Guarda ya Predictor+Analista, antes de intentar el Recomendador: si este falla,
        # no se pierde lo ya conseguido.
        resultados_pipeline[perfil] = registro
        with open(RUTA_RESULTADOS_PIPELINE, 'w', encoding='utf-8') as f:
            json.dump(resultados_pipeline, f, ensure_ascii=False, indent=2)

    # --- Etapa 3: Recomendador, solo si el router lo pide ---
    if ruta_info['ruta'] != 1:
        agente_elegido = (agente_recomendador_alto_valor if ruta_info['variante'] == 'alto_valor'
                           else agente_recomendador_cartera_masiva)

        tarea_recomendacion = crear_tarea_recomendacion(
            nombre_cliente=nombre_cliente,
            customer_data=customer_data,
            prediccion=registro['prediccion'],
            senales=registro['senales'],
            ruta_info=ruta_info,
            agente=agente_elegido,
            conclusion_analista=registro.get('conclusion_analista'),
            )
        crew_recomendador = Crew(agents=[agente_elegido], tasks=[tarea_recomendacion], process=Process.sequential)
        resultado_recomendacion = await ejecutar_con_reintentos(crew_recomendador)
        if resultado_recomendacion is None:
            print(f"⚠️ '{perfil}': Recomendador no completó tras los reintentos. "
                  f"Predictor+Analista quedan guardados; vuelve a llamar a esta función más tarde "
                  f"para reintentar solo esta etapa.")
        else:
            registro['recomendacion'] = resultado_recomendacion.pydantic.model_dump()
            uso_reco = getattr(resultado_recomendacion, 'token_usage', None)
            registro['tokens_recomendador'] = uso_reco.total_tokens if uso_reco else None
            print(f"  → Recomendación: {registro['recomendacion']['accion_principal']}")
    else:
        print("  → Ruta 1: se cierra sin Recomendador, por diseño.")

    resultados_pipeline[perfil] = registro
    with open(RUTA_RESULTADOS_PIPELINE, 'w', encoding='utf-8') as f:
        json.dump(resultados_pipeline, f, ensure_ascii=False, indent=2)
    print(f"Guardado en {RUTA_RESULTADOS_PIPELINE} (clave: '{perfil}')")

    return registro

In [ ]:
# Claves disponibles: 'balance_cero', 'balance_alto', 'multi_producto',
# 'inactivo_alemania', 'perfil_limpio', 'desacuerdo_franco'
resultado = await ejecutar_pipeline_cliente('inactivo_alemania')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Predictor de Churn                                                                               │
│                                                                                                                 │
│  Task: Cliente: O'Donnell.                                                                                      │
│  Datos del cliente: {'CreditScore': 525, 'Geography': 'Germany', 'Gender': 'Male', 'Age': 33, 'Tenure': 4,      │
│  'Balance': 131023.76, 'NumOfProducts': 2, 'HasCrCard': 0, 'IsActiveMember': 0, 'EstimatedSalary': 55072.93}    │
│                                                                                                                 │
│  Llama a la tool de predicción de churn con estos datos.No añadas ninguna explicación, justificación ni         │
│  mención a las variables del cliente.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool predecir_churn_de_cliente executed with result: Predicción: 0 (0=no abandona, 1=abandona) | Probabilidad de churn: 19.70%...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Agente Predictor de Churn                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "cliente": "O'Donnell",                                                                                      │
│    "churn_prediction": 0,                                                                                       │
│    "churn_probability_pct": 19.7                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Task: Analiza el perfil del cliente 'O'Donnell' frente al histórico de la base.                                │
│  Datos del cliente:                                                                                             │
│  - CreditScore: 525                                                                                             │
│  - Geography: Germany                                                                                           │
│  - Gender: Male                                                                                                 │
│  - Age: 33                                                                                                      │
│  - Tenure: 4                                                                                                    │
│  - Balance: 131023.76                                                                                           │
│  - NumOfProducts: 2                                                                                             │
│  - HasCrCard: 0                                                                                                 │
│  - IsActiveMember: 0                                                                                            │
│  - EstimatedSalary: 55072.93                                                                                    │
│                                                                                                                 │
│  Empieza por las variables de comportamiento del cliente con la entidad (productos contratados y estado de      │
│  actividad), que son las que más separan la base histórica. Añade después las variables de perfil (saldo y      │
│  contexto demográfico) solo si su resultado puede cambiar la lectura del caso. Responde en español.Al elegir    │
│  la orientación, pondera la intensidad de cada señal, no su número: una señal muy alejada de la media (por      │
│  ejemplo x4) pesa más que varias señales ligeramente por debajo. Una sola señal extrema puede determinar la     │
│  orientación.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Tool productos executed with result: Clientes con 2 producto(s) (n=3672): churn 7.4% vs 20.4% base → x0.36, muy por debajo de la media...
Tool actividad executed with result: Clientes inactivos y sin tarjeta (n=1119): churn 25.7% vs 20.4% base → x1.26, por encima de la media...
Tool balance executed with result: Saldo de 131,023.76 € → percentil 78.0 de la base. Clientes del tercer cuartil de saldo positivo (n=1277): churn 25.4% vs 20.4% base → x1.25, por encima de la media...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Negocio Senior                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "cliente": "O'Donnell",                                                                                      │
│    "senales": [                                                                                                 │
│      "productos: churn 7.4% vs 20.4% base → x0.36",                                                             │
│      "actividad: churn 25.7% vs 20.4% base → x1.26",                                                            │
│      "balance: churn 25.4% vs 20.4% base → x1.25"                                                               │
│    ],                                                                                                           │
│    "orientacion": "riesgo_por_encima_de_la_media",                                                              │
│    "conclusion": "El cliente presenta un riesgo de abandono superior al promedio, impulsado por su inactividad  │
│  y alto saldo, pese a la baja tasa de churn asociada a la contratación de dos productos."                       │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[inactivo_alemania] O'Donnell — Predictor 19.70% | Analista: riesgo_por_debajo_de_la_media | Ruta: cierre_sin_recomendador


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  → Ruta 1: se cierra sin Recomendador, por diseño.
Guardado en resultados_pipeline_completo.json (clave: 'inactivo_alemania')


## Evaluaciones

### 1. Utilidad recomendación con rúbrica

In [5]:
import json

def enrutar_tradicional(churn_probability_pct: float, balance: float) -> dict:
    """Modelo tradicional: decide la banda de riesgo usando SOLO la probabilidad del
    modelo, con los mismos umbrales que el router. No hay Analista, así que por debajo
    del 30 % el caso siempre se cierra. El percentil de saldo solo elige el canal."""
    percentil_valor = percentil_balance_numerico(balance)
    variante = "alto_valor" if percentil_valor >= UMBRAL_VALOR_ALTO else "cartera_masiva"

    if churn_probability_pct < UMBRAL_PROB_BAJA:
        return {'ruta': 1, 'nombre_ruta': 'cierre_sin_accion', 'variante': None}
    if churn_probability_pct >= UMBRAL_PROB_ALTA:
        return {'ruta': 2, 'nombre_ruta': 'riesgo_alto', 'variante': variante}
    return {'ruta': 3, 'nombre_ruta': 'riesgo_medio', 'variante': variante}


def recomendacion_tradicional(nombre_cliente: str, ruta_info: dict) -> dict:
    """Modelo tradicional de reglas fijas: una acción genérica por banda de riesgo, sin usar
    ninguna señal específica del cliente ni la simulación contrafactual. Línea base para medir
    cuánta utilidad añade el Recomendador agéntico."""
    ruta = ruta_info['ruta']

    if ruta == 1:
        accion = "Sin acción. El cliente no supera el umbral de riesgo del modelo."
        canal = None
    elif ruta == 2:
        accion = ("Contacto urgente de retención: ofrecer condiciones preferentes estándar "
                   "(mismo guion para todos los clientes de esta banda de riesgo).")
        canal = "gestor personal" if ruta_info['variante'] == 'alto_valor' else "contact center"
    else:  # ruta 3
        accion = ("Incluir en seguimiento de riesgo medio: contacto de revisión general, "
                   "sin oferta específica.")
        canal = "contact center"

    return {
        'cliente': nombre_cliente,
        'ruta': ruta_info['nombre_ruta'],
        'canal': canal,
        'accion_principal': accion,
        'palanca_objetivo': None,
        'justificacion': (f"Probabilidad de churn del modelo por encima o debajo de los "
                           f"umbrales fijos ({UMBRAL_PROB_BAJA}% / {UMBRAL_PROB_ALTA}%). "
                           f"No se han considerado señales adicionales del cliente."),
    }


# Si no se han reconstruido los casos en esta sesión, se cargan los congelados
if 'casos' not in globals():
    with open('casos_recomendador.json', encoding='utf-8') as f:
        casos = json.load(f)

baseline_tradicional = {}
for perfil, caso in casos.items():
    ruta_info = enrutar_tradicional(
        churn_probability_pct=caso['prediccion']['churn_probability_pct'],
        balance=caso['datos']['Balance'],
    )
    baseline_tradicional[perfil] = recomendacion_tradicional(caso['cliente'], ruta_info)

with open('baseline_tradicional.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_tradicional, f, ensure_ascii=False, indent=2)

for perfil, r in baseline_tradicional.items():
    print(f"\n=== {perfil} — {r['cliente']} ({r['ruta']}) ===")
    print(f"Canal: {r['canal']}")
    print(f"Acción: {r['accion_principal']}")


=== balance_cero — Hsiung (cierre_sin_accion) ===
Canal: None
Acción: Sin acción. El cliente no supera el umbral de riesgo del modelo.

=== balance_alto — Lo (riesgo_alto) ===
Canal: gestor personal
Acción: Contacto urgente de retención: ofrecer condiciones preferentes estándar (mismo guion para todos los clientes de esta banda de riesgo).

=== multi_producto — Gardiner (riesgo_alto) ===
Canal: contact center
Acción: Contacto urgente de retención: ofrecer condiciones preferentes estándar (mismo guion para todos los clientes de esta banda de riesgo).

=== inactivo_alemania — O'Donnell (cierre_sin_accion) ===
Canal: None
Acción: Sin acción. El cliente no supera el umbral de riesgo del modelo.

=== perfil_limpio — Sun (riesgo_medio) ===
Canal: contact center
Acción: Incluir en seguimiento de riesgo medio: contacto de revisión general, sin oferta específica.

=== desacuerdo_franco — Gray (cierre_sin_accion) ===
Canal: None
Acción: Sin acción. El cliente no supera el umbral de riesgo del m

### 2.1 Modelo tradicional vs sistema completo --> cobertura de riesgo

In [24]:
def comparar_cobertura_riesgo(X_test_raw: pd.DataFrame) -> pd.DataFrame:
    """Compara, sobre todo el test set, la decisión de un modelo tradicional de umbral único
    (actuar solo si churn_probability_pct >= UMBRAL_PROB_BAJA) frente a la decisión real del
    sistema completo ( que exige acuerdo del Analista para cerrar sin más). """
    filas = []
    for _, fila in X_test_raw.iterrows():
        caso = construir_caso(fila)
        prob = caso['prediccion']['churn_probability_pct']
        orientacion = caso['orientacion_calculada']

        decision_tradicional = 'sin_accion' if prob < UMBRAL_PROB_BAJA else 'accion'

        ruta = enrutar_cliente(
            churn_probability_pct=prob,
            orientacion_calculada=orientacion,
            balance=float(fila['Balance']),
        )
        decision_sistema = 'sin_accion' if ruta['ruta'] == 1 else 'accion'

        filas.append({
            'cliente': caso['cliente'],
            'probabilidad_pct': prob,
            'orientacion_calculada': orientacion,
            'decision_tradicional': decision_tradicional,
            'decision_sistema': decision_sistema,
            'ruta_sistema': ruta['nombre_ruta'],
        })

    return pd.DataFrame(filas)


resultado_cobertura = comparar_cobertura_riesgo(X_test_raw)

rescatados = resultado_cobertura[
    (resultado_cobertura['decision_tradicional'] == 'sin_accion')
    & (resultado_cobertura['decision_sistema'] == 'accion')
]

print(f"Clientes en el test set: {len(resultado_cobertura)}")
print(f"El modelo tradicional (con umbral único {UMBRAL_PROB_BAJA}%) cerraría sin acción: "
      f"{(resultado_cobertura['decision_tradicional'] == 'sin_accion').sum()}")
print(f"De ellos, el sistema completo los mantiene en revisión por desacuerdo del Analista: "
      f"{len(rescatados)} ({100 * len(rescatados) / len(resultado_cobertura):.2f}% del total)")

# csv de 2000 filas (todos los clientes de test) con las decisiones del tradicional y del sistema
# para cada uno
resultado_cobertura.to_csv('comparacion_cobertura_riesgo.csv', index=False)

Clientes en el test set: 2000
El modelo tradicional (umbral único en 30.0%) cerraría sin acción: 1053
De ellos, el sistema completo los mantiene en revisión por desacuerdo del Analista: 81 (4.05% del total)


### 2.2. Validación de los clientes rescatados con la etiqueta real

La prueba anterior se ve que el sistema mantiene en revision a 81 clientes, pero no se sabe si esos clientes en verdad hacen churn o no.

In [26]:
# Validación con la etiqueta real (y_test) para comporbar si los clientes que el sistema mantiene
# en revisión y el modelo tradicional cierra abandonan?
import numpy as np
from scipy.stats import fisher_exact

# 1. Etiqueta real, alineada por posición con X_test_raw
y_test_raw = pd.read_csv('y_test_raw.csv')
if 'resultado_cobertura' not in globals():
    resultado_cobertura = pd.read_csv('comparacion_cobertura_riesgo.csv')

val = resultado_cobertura.copy()
val['Exited'] = y_test_raw['Exited'].astype(int).values #creamos nueva columna exited con el valor exited de cada uno

# Comprobamos que las filas están en el mismo orden que X_test_raw
assert (val['cliente'].values == X_test_raw['Surname'].values).all(), "Filas desalineadas"

# 2. Grupos
trad_cierra = val['decision_tradicional'] == 'sin_accion'
sist_cierra = val['decision_sistema'] == 'sin_accion'
rescatados = val[trad_cierra & ~sist_cierra]    # el sistema los mantiene en revisión
ambos_cierran = val[trad_cierra & sist_cierra]  # el sistema también los cierra

# 3. calcular tasa real de abandono de cada grupo
tabla_validacion = pd.DataFrame([
    {'grupo': nombre, 'clientes': len(g), 'abandonan': int(g['Exited'].sum()),
     'tasa_abandono_%': round(100 * g['Exited'].sum() / len(g), 1)}
    for nombre, g in [('Rescatados por el sistema', rescatados),
                      ('Ambos cierran (Ruta 1)', ambos_cierran),
                      ('Global test set', val)]
])
print(tabla_validacion.to_string(index=False))

# 4. ¿La diferencia puede deberse al azar? Test exacto de Fisher (adecuado con pocos casos)
_, p_valor = fisher_exact(
    [[rescatados['Exited'].sum(), (rescatados['Exited'] == 0).sum()],
     [ambos_cierran['Exited'].sum(), (ambos_cierran['Exited'] == 0).sum()]],
    alternative='greater')
print(f"\np-valor (rescatados abandonan más que ambos_cierran): {p_valor:.4f}")

tabla_validacion.to_csv('tabla_validacion_rescatados.csv', index=False)

                    grupo  clientes  abandonan  tasa_abandono_%
Rescatados por el sistema        81         10             12.3
   Ambos cierran (Ruta 1)       972         42              4.3
          Global test set      2000        407             20.4

Fisher unilateral (rescatados > ambos cierran): odds ratio = 3.12, p = 0.0045

p-valor (rescatados abandonan más que ambos_cierran): 0.0045


## Guardar versiones

In [ ]:
#Guardamos las versiones utilizadaS
!pip freeze | grep -iE "crewai|litellm|groq|pydantic|pandas|numpy|scikit-learn|lightgbm|joblib" > requirements_agente_groq.txt
!cat requirements_agente_groq.txt

crewai==1.15.21
crewai-cli==1.15.21
crewai-core==1.15.21
geopandas==1.1.4
joblib==1.6.0
lightgbm==4.6.0
litellm==1.101.0
numpy==2.1.3
pandas==2.2.3
pandas-datareader==0.11.1
pandas-gbq==0.30.0
pandas-stubs==2.2.3.250527
pydantic==2.12.5
pydantic-settings==2.15.0
pydantic_core==2.41.5
scikit-learn==1.6.1
sklearn-pandas==2.2.0
